# Week 7 Regime-Aware Supervised LSTM Autoencoder

This notebook starts from Hannah's supervised LSTM autoencoder and adds explicit robustness controls for volatile and drawdown-heavy regimes. The goal is to preserve upside capture while making validation, scoring, portfolio construction, and stress diagnostics regime-aware.

In [1]:
# Bootstrap: works locally, in Colab, or when repo_root is set manually.
import os
import sys
import subprocess
import tempfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
DEFAULT_REPO_URL = "https://github.com/halloyy/Portfolio-Optimization-Lib.git"


def _is_repo_root(path: Path) -> bool:
    path = Path(path).expanduser()
    return (path / "pyproject.toml").exists() and (path / "src" / "portfolio_toolkit").exists()


def _candidate_roots() -> list[Path]:
    candidates: list[Path] = []
    if "repo_root" in globals():
        candidates.append(Path(repo_root).expanduser())
    if os.environ.get("PORTFOLIO_REPO_ROOT"):
        candidates.append(Path(os.environ["PORTFOLIO_REPO_ROOT"]).expanduser())
    candidates.extend([Path.cwd(), *Path.cwd().parents])
    candidates.extend(
        [
            Path("/content/Portfolio-Optimization-Lib"),
            Path("/content/Portfolio-Optimizer"),
            Path("/workspace/Portfolio-Optimization-Lib"),
            Path("/Users/hannahlee/Documents/MLSN/Portfolio-Optimization-Lib"),
        ]
    )
    return candidates


def _find_repo_root() -> Path | None:
    for candidate in _candidate_roots():
        if _is_repo_root(candidate):
            return candidate.resolve()
    return None


repo_root = _find_repo_root()

if repo_root is None and IN_COLAB:
    repo_url = os.environ.get("PORTFOLIO_REPO_URL", DEFAULT_REPO_URL)
    clone_target = Path(os.environ.get("PORTFOLIO_REPO_ROOT", "/content/Portfolio-Optimization-Lib")).expanduser()
    if not clone_target.exists():
        subprocess.run(["git", "clone", repo_url, str(clone_target)], check=True)
    repo_root = clone_target.resolve() if _is_repo_root(clone_target) else None
    if repo_root is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_root)], check=True)

if repo_root is None:
    raise RuntimeError(
        "Cannot find repo root. Run this notebook from inside Portfolio-Optimization-Lib, "
        "or set repo_root = Path('/path/to/Portfolio-Optimization-Lib') before this cell."
    )

os.chdir(repo_root)
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "portfolio_matplotlib_cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("repo_root =", repo_root)
print("python    =", sys.executable)

repo_root = /content/Portfolio-Optimization-Lib
python    = /usr/bin/python3


In [2]:
import json
import math
from dataclasses import replace
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F

from portfolio_toolkit import (
    PortfolioWeights,
    backtest_weights,
    build_features,
    build_metrics,
    get_dataset_spec,
    init_mlflow,
    load_prices,
    log_backtest,
    log_model_submission,
    log_portfolio,
    log_predictions,
    make_forward_alpha_target,
    make_forward_return_target,
    split_dates,
    start_run,
    validate_prediction_frame,
    validate_weights_frame,
    write_backtest_artifacts,
)

warnings.filterwarnings("ignore")
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

torch: 2.10.0+cpu | cuda: False


## Configuration

In [3]:
DATASET_NAME = os.environ.get("DATASET_NAME", "shared_set_2")
MODEL_NAME = "hannah_week7_regime_lstm_autoencoder"
RUN_NAME = "Hannah_Week7_Regime_LSTM_AE"
HORIZON = 5
SEQ_LEN = 20
REBALANCE_FREQUENCY = "weekly"

LATENT_DIM = 32
HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = int(os.environ.get("LSTM_AE_EPOCHS", "18"))
DATES_PER_BATCH = 16
PATIENCE = 5
MASK_RATE = 0.10
GRAD_CLIP = 1.0

RECON_WEIGHT = 1.00
ALPHA_WEIGHT = 2.00
RETURN_WEIGHT = 0.25
RANK_WEIGHT = 0.30
VOL_WEIGHT = 0.35
DOWNSIDE_WEIGHT = 0.60
REGIME_WEIGHT = 0.30
HIGH_RISK_SAMPLE_MULTIPLIER = 1.50

BASE_TOP_FRACTION = 0.50
BASE_MAX_WEIGHT = 0.10
DYNAMIC_MIN_MAX_WEIGHT = 0.04
TURNOVER_BLEND = 0.50
BETA_TARGET = 1.00
BETA_SCORE_PENALTY = 0.15
DOWNSIDE_SCORE_PENALTY = 0.40
VOL_SCORE_PENALTY = 0.15
REGIME_SCORE_BONUS = 0.12

REGIME_CLASSES = ["calm_risk_on", "high_vol", "drawdown", "rebound"]
REGIME_CLASS_TO_ID = {name: idx for idx, name in enumerate(REGIME_CLASSES)}

DEFAULT_VARIANTS = "alpha_only,alpha_vol,alpha_downside,alpha_vol_regime"
VARIANT_NAMES = [name.strip() for name in os.environ.get("LSTM_AE_VARIANTS", DEFAULT_VARIANTS).split(",") if name.strip()]
FINAL_VARIANT_NAME = os.environ.get("FINAL_VARIANT_NAME", "alpha_vol_regime")
SELECT_BEST_VARIANT = os.environ.get("SELECT_BEST_VARIANT", "0") == "1"

RANDOM_SEED = 99
LOG_TO_MLFLOW = os.environ.get("SKIP_MLFLOW", "0") != "1"

output_dir = repo_root / "runs" / "supervised_lstm_autoencoder_week7"
output_dir.mkdir(parents=True, exist_ok=True)
NOTEBOOK_PATH = repo_root / "MODELS" / "Hannah" / "supervised_lstm_autoencoder_week7.ipynb"

spec = get_dataset_spec(DATASET_NAME, repo_root=repo_root)
tradable_tickers = [ticker for ticker in spec.tickers if ticker != spec.benchmark_ticker]
backtest_spec = replace(spec, tickers=tradable_tickers, dataset_id=DATASET_NAME)
splits = split_dates(DATASET_NAME, repo_root=repo_root)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
torch.backends.cudnn.deterministic = True

print("Dataset preset:", DATASET_NAME)
print("Dataset display name:", spec.name)
print("Tickers in preset:", len(spec.tickers))
print("Tradable tickers:", len(tradable_tickers))
print("Benchmark ticker:", spec.benchmark_ticker)
print("Train/Val/Test windows:", splits)
print("Epochs:", EPOCHS, "| variants:", VARIANT_NAMES, "| MLflow logging:", LOG_TO_MLFLOW)

Dataset preset: shared_set_2
Dataset display name: growth_tech_innovation
Tickers in preset: 26
Tradable tickers: 25
Benchmark ticker: SPY
Train/Val/Test windows: {'train': (Timestamp('2014-01-02 00:00:00'), Timestamp('2019-12-31 00:00:00')), 'val': (Timestamp('2020-01-02 00:00:00'), Timestamp('2021-12-31 00:00:00')), 'test': (Timestamp('2022-01-03 00:00:00'), Timestamp('2025-12-31 00:00:00'))}
Epochs: 18 | variants: ['alpha_only', 'alpha_vol', 'alpha_downside', 'alpha_vol_regime'] | MLflow logging: True


## Regime And Event-Risk Features

In [4]:
BASE_FEATURE_NAMES = [
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "vol_5d",
    "vol_20d",
    "vol_60d",
    "downside_vol_20d",
    "atr_14",
    "momentum_5d",
    "momentum_20d",
    "momentum_60d",
    "momentum_120d",
    "rsi_14",
    "macd_hist",
    "bollinger_z_20d",
    "price_to_sma_20d",
    "price_to_sma_50d",
    "price_to_sma_200d",
    "volume_zscore_20d",
    "dollar_volume_ratio_20d",
    "intraday_range",
    "beta_20d_spy",
    "beta_60d_spy",
    "excess_return_20d_vs_spy",
    "excess_return_60d_vs_spy",
    "relative_momentum_20d_vs_spy",
]

CUSTOM_FEATURE_NAMES = [
    "mom_vol_ratio_60d",
    "trend_spread_20_50",
    "trend_spread_50_200",
    "rolling_sharpe_60d",
    "drawdown_60d",
    "spy_vol_20d_ann",
    "spy_momentum_20d",
    "spy_momentum_60d",
    "spy_drawdown_60d",
    "market_breadth_20d",
    "pct_above_sma50",
    "pct_above_sma200",
    "pct_20d_highs",
    "pct_20d_lows",
    "avg_universe_drawdown_60d",
    "cross_sectional_dispersion_20d",
    "gap_frequency_20d",
    "volume_shock_20d",
    "abnormal_range_20d",
    "vol_compression_ratio_20_60",
    "distance_from_60d_high",
    "momentum_x_spy_vol",
    "momentum_x_spy_drawdown",
    "beta_instability_60d",
]

ALL_FEATURE_NAMES = BASE_FEATURE_NAMES + CUSTOM_FEATURE_NAMES
REGIME_CONTEXT_COLUMNS = ["spy_vol_20d_ann", "spy_momentum_20d", "spy_momentum_60d", "spy_drawdown_60d"]


def _safe_pct_change(series: pd.Series, periods: int = 1) -> pd.Series:
    return series.astype(float).pct_change(periods)


def build_model_features(prices: pd.DataFrame) -> pd.DataFrame:
    """Rebuild the exact Week 7 LSTM-AE feature table from raw long-form prices."""
    frame = build_features(prices, feature_names=BASE_FEATURE_NAMES)
    panel = prices.copy()
    panel["date"] = pd.to_datetime(panel["date"], utc=True).dt.tz_localize(None)
    panel["ticker"] = panel["ticker"].astype(str).str.upper()
    panel = panel.sort_values(["ticker", "date"]).reset_index(drop=True)
    eps = 1e-6

    frame["mom_vol_ratio_60d"] = frame["momentum_60d"] / (frame["vol_60d"].abs() + eps)
    frame["trend_spread_20_50"] = frame["price_to_sma_20d"] - frame["price_to_sma_50d"]
    frame["trend_spread_50_200"] = frame["price_to_sma_50d"] - frame["price_to_sma_200d"]
    frame["rolling_sharpe_60d"] = frame["return_60d"] / (frame["vol_60d"].abs() * np.sqrt(60.0) + eps)

    grouped_close = panel.groupby("ticker", sort=False)["adj_close"]
    rolling_high_20 = grouped_close.transform(lambda s: s.rolling(20, min_periods=20).max())
    rolling_low_20 = grouped_close.transform(lambda s: s.rolling(20, min_periods=20).min())
    rolling_high_60 = grouped_close.transform(lambda s: s.rolling(60, min_periods=20).max())
    prev_close = grouped_close.shift(1)

    event = panel.loc[:, ["date", "ticker"]].copy()
    event["drawdown_60d"] = panel["adj_close"] / rolling_high_60.replace(0.0, np.nan) - 1.0
    event["distance_from_60d_high"] = panel["adj_close"] / rolling_high_60.replace(0.0, np.nan) - 1.0
    event["gap_abs"] = (panel["open"] / prev_close.replace(0.0, np.nan) - 1.0).abs()
    event["range_ratio"] = (panel["high"] - panel["low"]) / panel["adj_close"].replace(0.0, np.nan)
    event["gap_frequency_20d"] = event.groupby(panel["ticker"], sort=False)["gap_abs"].transform(
        lambda s: (s > 0.03).rolling(20, min_periods=10).mean()
    )
    event["abnormal_range_20d"] = event["range_ratio"] / (
        event.groupby(panel["ticker"], sort=False)["range_ratio"].transform(lambda s: s.rolling(20, min_periods=10).mean()) + eps
    )
    rolling_vol_20 = grouped_close.transform(lambda s: _safe_pct_change(s).rolling(20, min_periods=20).std(ddof=0))
    rolling_vol_60 = grouped_close.transform(lambda s: _safe_pct_change(s).rolling(60, min_periods=30).std(ddof=0))
    event["vol_compression_ratio_20_60"] = rolling_vol_20 / (rolling_vol_60 + eps)
    volume_ma_20 = panel.groupby("ticker", sort=False)["volume"].transform(lambda s: s.rolling(20, min_periods=10).mean())
    event["volume_shock_20d"] = panel["volume"] / volume_ma_20.replace(0.0, np.nan)
    event = event.drop(columns=["gap_abs", "range_ratio"])
    frame = frame.merge(event, on=["date", "ticker"], how="left")

    spy = panel.loc[panel["ticker"] == "SPY", ["date", "adj_close"]].sort_values("date").copy()
    if spy.empty:
        regime = pd.DataFrame(
            {
                "date": pd.to_datetime(panel["date"].drop_duplicates()),
                "spy_vol_20d_ann": np.nan,
                "spy_momentum_20d": np.nan,
                "spy_momentum_60d": np.nan,
                "spy_drawdown_60d": np.nan,
            }
        )
    else:
        spy_ret = spy["adj_close"].pct_change()
        regime = spy.loc[:, ["date"]].copy()
        regime["spy_vol_20d_ann"] = spy_ret.rolling(20, min_periods=20).std(ddof=0) * np.sqrt(252.0)
        regime["spy_momentum_20d"] = spy["adj_close"].pct_change(20)
        regime["spy_momentum_60d"] = spy["adj_close"].pct_change(60)
        spy_high_60 = spy["adj_close"].rolling(60, min_periods=20).max()
        regime["spy_drawdown_60d"] = spy["adj_close"] / spy_high_60.replace(0.0, np.nan) - 1.0
    frame = frame.merge(regime, on="date", how="left")

    non_benchmark = panel.loc[panel["ticker"] != "SPY"].copy()
    nb_close = non_benchmark.groupby("ticker", sort=False)["adj_close"]
    nb_return = nb_close.pct_change()
    sma_20 = nb_close.transform(lambda s: s.rolling(20, min_periods=20).mean())
    sma_50 = nb_close.transform(lambda s: s.rolling(50, min_periods=30).mean())
    sma_200 = nb_close.transform(lambda s: s.rolling(200, min_periods=100).mean())
    high_20 = nb_close.transform(lambda s: s.rolling(20, min_periods=20).max())
    low_20 = nb_close.transform(lambda s: s.rolling(20, min_periods=20).min())
    high_60_nb = nb_close.transform(lambda s: s.rolling(60, min_periods=20).max())
    breadth_panel = non_benchmark.loc[:, ["date", "ticker"]].copy()
    breadth_panel["above_sma20"] = non_benchmark["adj_close"] > sma_20
    breadth_panel["above_sma50"] = non_benchmark["adj_close"] > sma_50
    breadth_panel["above_sma200"] = non_benchmark["adj_close"] > sma_200
    breadth_panel["is_20d_high"] = non_benchmark["adj_close"] >= high_20
    breadth_panel["is_20d_low"] = non_benchmark["adj_close"] <= low_20
    breadth_panel["universe_drawdown_60d"] = non_benchmark["adj_close"] / high_60_nb.replace(0.0, np.nan) - 1.0
    breadth_panel["daily_return"] = nb_return.to_numpy(dtype=float)

    breadth = breadth_panel.groupby("date").agg(
        market_breadth_20d=("above_sma20", "mean"),
        pct_above_sma50=("above_sma50", "mean"),
        pct_above_sma200=("above_sma200", "mean"),
        pct_20d_highs=("is_20d_high", "mean"),
        pct_20d_lows=("is_20d_low", "mean"),
        avg_universe_drawdown_60d=("universe_drawdown_60d", "mean"),
        cross_sectional_dispersion_20d=("daily_return", "std"),
    ).reset_index()
    breadth["cross_sectional_dispersion_20d"] = breadth["cross_sectional_dispersion_20d"].rolling(20, min_periods=10).mean()
    frame = frame.merge(breadth, on="date", how="left")

    frame["momentum_x_spy_vol"] = frame["momentum_20d"] * frame["spy_vol_20d_ann"]
    frame["momentum_x_spy_drawdown"] = frame["momentum_20d"] * frame["spy_drawdown_60d"]
    frame["beta_instability_60d"] = frame.groupby("ticker", sort=False)["beta_20d_spy"].transform(
        lambda s: s.rolling(60, min_periods=20).std(ddof=0)
    )

    return frame.loc[:, ["date", "ticker"] + ALL_FEATURE_NAMES]

In [5]:
prices = load_prices(DATASET_NAME, repo_root=repo_root)
print("Price frame shape:", prices.shape)
print("Date range:", prices["date"].min(), "->", prices["date"].max())
print("Unique tickers:", prices["ticker"].nunique())

feature_frame = build_model_features(prices)
print("Feature frame shape:", feature_frame.shape)
print("Feature count:", len(ALL_FEATURE_NAMES))
display(feature_frame.head())

Price frame shape: (78468, 8)
Date range: 2014-01-02 00:00:00 -> 2025-12-31 00:00:00
Unique tickers: 26
Feature frame shape: (78468, 53)
Feature count: 51


,date,ticker,return_1d,return_5d,return_20d,return_60d,vol_5d,vol_20d,vol_60d,downside_vol_20d,...,avg_universe_drawdown_60d,cross_sectional_dispersion_20d,gap_frequency_20d,volume_shock_20d,abnormal_range_20d,vol_compression_ratio_20_60,distance_from_60d_high,momentum_x_spy_vol,momentum_x_spy_drawdown,beta_instability_60d
0,2014-01-02,AAPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2014-01-03,AAPL,-0.021966,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2014-01-06,AAPL,0.005453,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2014-01-07,AAPL,-0.007151,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2014-01-08,AAPL,0.006332,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Targets, Regimes, And Sequences

In [6]:
return_target_col = f"forward_return_{HORIZON}d"
alpha_target_col = f"forward_alpha_{HORIZON}d_vs_spy"
forward_vol_target_col = f"forward_realized_vol_{HORIZON}d"
bottom_quintile_col = f"bottom_return_quintile_{HORIZON}d"


def make_forward_realized_vol_target(prices: pd.DataFrame, horizon: int) -> pd.DataFrame:
    panel = prices.copy()
    panel["date"] = pd.to_datetime(panel["date"], utc=True).dt.tz_localize(None)
    panel["ticker"] = panel["ticker"].astype(str).str.upper()
    panel = panel.sort_values(["ticker", "date"]).reset_index(drop=True)
    panel["daily_return"] = panel.groupby("ticker", sort=False)["adj_close"].pct_change()

    def _future_vol(ret: pd.Series) -> pd.Series:
        shifted = ret.shift(-1)
        return shifted.iloc[::-1].rolling(horizon, min_periods=horizon).std(ddof=0).iloc[::-1] * np.sqrt(252.0)

    panel[forward_vol_target_col] = panel.groupby("ticker", sort=False)["daily_return"].transform(_future_vol)
    return panel.loc[:, ["date", "ticker", forward_vol_target_col]]


def make_regime_thresholds(feature_frame: pd.DataFrame, train_start, train_end) -> dict[str, float]:
    context = feature_frame.loc[:, ["date"] + REGIME_CONTEXT_COLUMNS].drop_duplicates("date").copy()
    dates = pd.to_datetime(context["date"])
    train_context = context.loc[(dates >= pd.Timestamp(train_start)) & (dates <= pd.Timestamp(train_end))].replace([np.inf, -np.inf], np.nan)
    vol_threshold = float(train_context["spy_vol_20d_ann"].quantile(0.75))
    drawdown_threshold = float(min(-0.05, train_context["spy_drawdown_60d"].quantile(0.25)))
    rebound_momentum_threshold = float(max(0.0, train_context["spy_momentum_20d"].median()))
    rally_threshold = float(train_context["spy_momentum_20d"].quantile(0.75))
    return {
        "high_vol_spy_vol_20d_ann": vol_threshold,
        "drawdown_spy_drawdown_60d": drawdown_threshold,
        "rebound_spy_momentum_20d": rebound_momentum_threshold,
        "rally_spy_momentum_20d": rally_threshold,
    }


def assign_market_regime(frame: pd.DataFrame, thresholds: dict[str, float]) -> pd.Series:
    vol = pd.to_numeric(frame["spy_vol_20d_ann"], errors="coerce")
    drawdown = pd.to_numeric(frame["spy_drawdown_60d"], errors="coerce")
    momentum = pd.to_numeric(frame["spy_momentum_20d"], errors="coerce")
    labels = pd.Series("calm_risk_on", index=frame.index, dtype="object")
    labels.loc[vol >= thresholds["high_vol_spy_vol_20d_ann"]] = "high_vol"
    labels.loc[drawdown <= thresholds["drawdown_spy_drawdown_60d"]] = "drawdown"
    rebound = (drawdown <= thresholds["drawdown_spy_drawdown_60d"]) & (momentum > thresholds["rebound_spy_momentum_20d"])
    labels.loc[rebound] = "rebound"
    return labels


return_targets = make_forward_return_target(prices, horizon=HORIZON)
alpha_targets = make_forward_alpha_target(prices, horizon=HORIZON)
vol_targets = make_forward_realized_vol_target(prices, horizon=HORIZON)
regime_thresholds = make_regime_thresholds(feature_frame, spec.train_start, spec.train_end)

feature_frame = feature_frame.copy()
feature_frame["market_regime"] = assign_market_regime(feature_frame, regime_thresholds)
feature_frame["market_regime_code"] = feature_frame["market_regime"].map(REGIME_CLASS_TO_ID).astype(int)
feature_frame["is_high_risk_regime"] = feature_frame["market_regime"].isin(["high_vol", "drawdown"]).astype(float)

target_frame = feature_frame.merge(return_targets, on=["date", "ticker"], how="left")
target_frame = target_frame.merge(alpha_targets, on=["date", "ticker"], how="left")
target_frame = target_frame.merge(vol_targets, on=["date", "ticker"], how="left")
target_frame = target_frame.replace([np.inf, -np.inf], np.nan)
target_frame[bottom_quintile_col] = (
    target_frame.groupby("date")[return_target_col].rank(method="average", pct=True) <= 0.20
).astype(float)
target_frame = (
    target_frame.dropna(subset=ALL_FEATURE_NAMES + [return_target_col, alpha_target_col, forward_vol_target_col])
    .sort_values(["ticker", "date"])
    .reset_index(drop=True)
)
target_frame["historical_vol_20d"] = np.clip(target_frame["vol_20d"].to_numpy(dtype=float) * np.sqrt(252.0), 1e-4, None)
target_frame[forward_vol_target_col] = np.clip(target_frame[forward_vol_target_col].to_numpy(dtype=float), 1e-4, None)
target_frame["forward_vol_log"] = np.log1p(target_frame[forward_vol_target_col])


def _split_mask(frame: pd.DataFrame, start, end) -> pd.Series:
    dates = pd.to_datetime(frame["date"])
    return (dates >= pd.Timestamp(start)) & (dates <= pd.Timestamp(end))


train_rows = target_frame.loc[_split_mask(target_frame, spec.train_start, spec.train_end)].copy()
val_rows = target_frame.loc[_split_mask(target_frame, spec.val_start, spec.val_end)].copy()
test_rows = target_frame.loc[_split_mask(target_frame, spec.test_start, spec.test_end)].copy()
train_model_rows = train_rows.loc[train_rows["ticker"] != spec.benchmark_ticker].copy()

train_means = train_model_rows[ALL_FEATURE_NAMES].mean()
train_stds = train_model_rows[ALL_FEATURE_NAMES].std(ddof=0).replace(0.0, 1.0)

target_means = {
    "return": float(train_model_rows[return_target_col].mean()),
    "alpha": float(train_model_rows[alpha_target_col].mean()),
    "vol_log": float(train_model_rows["forward_vol_log"].mean()),
}
target_stds = {
    "return": float(train_model_rows[return_target_col].std(ddof=0) or 1.0),
    "alpha": float(train_model_rows[alpha_target_col].std(ddof=0) or 1.0),
    "vol_log": float(train_model_rows["forward_vol_log"].std(ddof=0) or 1.0),
}


def standardize_features(frame: pd.DataFrame) -> np.ndarray:
    return ((frame[ALL_FEATURE_NAMES] - train_means) / train_stds).to_numpy(dtype=np.float32)


X_all = standardize_features(target_frame)

print("Modeling frame:", target_frame.shape)
print("Train/Val/Test rows:", len(train_rows), len(val_rows), len(test_rows))
print("Train rows excluding benchmark:", len(train_model_rows))
print("Feature count:", len(ALL_FEATURE_NAMES))
print("Regime thresholds:", json.dumps(regime_thresholds, indent=2, sort_keys=True))
print("Regime distribution:")
display(target_frame.drop_duplicates(["date"])["market_regime"].value_counts(normalize=True).rename("share").to_frame())

Modeling frame: (73151, 62)
Train/Val/Test rows: 34081 13130 25940
Train rows excluding benchmark: 32770
Feature count: 51
Regime thresholds: {
  "drawdown_spy_drawdown_60d": -0.05,
  "high_vol_spy_vol_20d_ann": 0.14315739691564988,
  "rally_spy_momentum_20d": 0.028799591057829477,
  "rebound_spy_momentum_20d": 0.012521388300228753
}
Regime distribution:


,share
market_regime,
calm_risk_on,0.612651
high_vol,0.186923
drawdown,0.165245
rebound,0.035181


In [7]:
def make_model_sequences(frame: pd.DataFrame, X: np.ndarray, seq_len: int) -> tuple[np.ndarray, pd.DataFrame]:
    Xs: list[np.ndarray] = []
    meta: list[dict[str, object]] = []
    for ticker, group in frame.groupby("ticker", sort=False):
        positions = group.index.to_numpy(dtype=int)
        if len(positions) < seq_len:
            continue
        dates = pd.to_datetime(group["date"]).to_numpy()
        for end_offset in range(seq_len - 1, len(positions)):
            window_positions = positions[end_offset - seq_len + 1 : end_offset + 1]
            row = group.iloc[end_offset]
            Xs.append(X[window_positions])
            meta.append(
                {
                    "date": pd.Timestamp(dates[end_offset]),
                    "ticker": ticker,
                    "row_position": int(positions[end_offset]),
                    "target_return": float(row[return_target_col]),
                    "target_alpha": float(row[alpha_target_col]),
                    "target_forward_vol": float(row[forward_vol_target_col]),
                    "target_vol_log": float(row["forward_vol_log"]),
                    "target_bottom_quintile": float(row[bottom_quintile_col]),
                    "market_regime": str(row["market_regime"]),
                    "market_regime_code": int(row["market_regime_code"]),
                    "is_high_risk_regime": float(row["is_high_risk_regime"]),
                    "historical_vol_20d": float(row["historical_vol_20d"]),
                    "spy_vol_20d_ann": float(row["spy_vol_20d_ann"]),
                    "spy_drawdown_60d": float(row["spy_drawdown_60d"]),
                    "spy_momentum_20d": float(row["spy_momentum_20d"]),
                    "beta_60d_spy": float(row["beta_60d_spy"]),
                    "drawdown_60d": float(row["drawdown_60d"]),
                    "market_breadth_20d": float(row["market_breadth_20d"]),
                }
            )
    return np.stack(Xs).astype(np.float32), pd.DataFrame(meta)


X_seq, seq_meta = make_model_sequences(target_frame, X_all, SEQ_LEN)
seq_meta["return_scaled"] = ((seq_meta["target_return"] - target_means["return"]) / target_stds["return"]).astype(np.float32)
seq_meta["alpha_scaled"] = ((seq_meta["target_alpha"] - target_means["alpha"]) / target_stds["alpha"]).astype(np.float32)
seq_meta["vol_scaled"] = ((seq_meta["target_vol_log"] - target_means["vol_log"]) / target_stds["vol_log"]).astype(np.float32)
seq_meta["sample_weight"] = 1.0 + (HIGH_RISK_SAMPLE_MULTIPLIER - 1.0) * seq_meta["is_high_risk_regime"].astype(float)

non_benchmark_seq = seq_meta["ticker"] != spec.benchmark_ticker
train_seq_mask = _split_mask(seq_meta, spec.train_start, spec.train_end) & non_benchmark_seq
val_seq_mask = _split_mask(seq_meta, spec.val_start, spec.val_end) & non_benchmark_seq
test_seq_mask = _split_mask(seq_meta, spec.test_start, spec.test_end) & non_benchmark_seq

X_train_seq = X_seq[train_seq_mask.to_numpy()]
X_val_seq = X_seq[val_seq_mask.to_numpy()]
X_test_seq_all = X_seq[test_seq_mask.to_numpy()]

meta_train = seq_meta.loc[train_seq_mask].reset_index(drop=True)
meta_val = seq_meta.loc[val_seq_mask].reset_index(drop=True)
meta_test_all = seq_meta.loc[test_seq_mask].reset_index(drop=True)

y_train_return = meta_train["return_scaled"].to_numpy(dtype=np.float32)
y_train_alpha = meta_train["alpha_scaled"].to_numpy(dtype=np.float32)
y_train_vol = meta_train["vol_scaled"].to_numpy(dtype=np.float32)
y_train_downside = meta_train["target_bottom_quintile"].to_numpy(dtype=np.float32)
y_train_regime = meta_train["market_regime_code"].to_numpy(dtype=np.int64)
w_train = meta_train["sample_weight"].to_numpy(dtype=np.float32)

y_val_return = meta_val["return_scaled"].to_numpy(dtype=np.float32)
y_val_alpha = meta_val["alpha_scaled"].to_numpy(dtype=np.float32)
y_val_vol = meta_val["vol_scaled"].to_numpy(dtype=np.float32)
y_val_downside = meta_val["target_bottom_quintile"].to_numpy(dtype=np.float32)
y_val_regime = meta_val["market_regime_code"].to_numpy(dtype=np.int64)
w_val = meta_val["sample_weight"].to_numpy(dtype=np.float32)

print("All sequences:", X_seq.shape)
print("Train sequences:", X_train_seq.shape)
print("Val sequences:", X_val_seq.shape)
print("Test sequences:", X_test_seq_all.shape)

All sequences: (72657, 20, 51)
Train sequences: (32295, 20, 51)
Val sequences: (12625, 20, 51)
Test sequences: (24942, 20, 51)


## Model, Multi-Task Losses, And Variant Sweep

In [8]:
class RegimeAwareSupervisedLSTMAutoencoder(nn.Module):
    def __init__(self, input_dim: int, hidden_size: int, latent_dim: int, num_layers: int, dropout: float, regime_classes: int):
        super().__init__()
        self.input_dim = int(input_dim)
        self.hidden_size = int(hidden_size)
        self.latent_dim = int(latent_dim)
        self.num_layers = int(num_layers)
        self.dropout = float(dropout)
        self.regime_classes = int(regime_classes)
        self.encoder = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.to_latent = nn.Sequential(
            nn.Linear(hidden_size, latent_dim),
            nn.LayerNorm(latent_dim),
            nn.Tanh(),
        )
        self.decoder_seed = nn.Linear(latent_dim, hidden_size)
        self.decoder = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
        )
        self.reconstruction_head = nn.Linear(hidden_size, input_dim)
        self.return_head = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))
        self.alpha_head = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))
        self.vol_head = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))
        self.downside_head = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))
        self.regime_head = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, regime_classes))

    def forward(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        _, (h_n, _) = self.encoder(x)
        latent = self.to_latent(h_n[-1])
        repeated = self.decoder_seed(latent).unsqueeze(1).repeat(1, x.shape[1], 1)
        decoded, _ = self.decoder(repeated)
        return {
            "reconstruction": self.reconstruction_head(decoded),
            "expected_return": self.return_head(latent).squeeze(-1),
            "expected_alpha": self.alpha_head(latent).squeeze(-1),
            "expected_vol_log": self.vol_head(latent).squeeze(-1),
            "downside_logit": self.downside_head(latent).squeeze(-1),
            "regime_logits": self.regime_head(latent),
            "latent": latent,
        }


def make_reconstruction_feature_weights(feature_names: list[str]) -> pd.Series:
    weights = pd.Series(1.0, index=feature_names, dtype=float)
    emphasis_terms = ["return", "vol", "downside", "drawdown", "spy_", "breadth", "dispersion", "beta", "range"]
    for name in feature_names:
        if any(term in name for term in emphasis_terms):
            weights.loc[name] = 1.75
    return weights / weights.mean()


reconstruction_feature_weights = make_reconstruction_feature_weights(ALL_FEATURE_NAMES)
reconstruction_weight_tensor = torch.as_tensor(reconstruction_feature_weights.to_numpy(dtype=np.float32)).view(1, 1, -1)


def apply_feature_denoising(x: torch.Tensor, mask_rate: float) -> torch.Tensor:
    if mask_rate <= 0.0:
        return x
    keep = (torch.rand_like(x) > mask_rate).to(x.dtype)
    return x * keep


def weighted_mse(pred: torch.Tensor, target: torch.Tensor, sample_weight: torch.Tensor) -> torch.Tensor:
    return (sample_weight * (pred - target).pow(2)).mean()


def pairwise_rank_loss(pred_alpha: torch.Tensor, true_alpha: torch.Tensor, date_codes: torch.Tensor) -> torch.Tensor:
    losses = []
    for date_code in torch.unique(date_codes):
        idx = torch.where(date_codes == date_code)[0]
        if idx.numel() < 2:
            continue
        pred = pred_alpha[idx]
        target = true_alpha[idx]
        target_diff = target[:, None] - target[None, :]
        useful_pairs = target_diff > 0
        if useful_pairs.any():
            pred_diff = pred[:, None] - pred[None, :]
            losses.append(F.softplus(-pred_diff[useful_pairs]).mean())
    if not losses:
        return pred_alpha.new_tensor(0.0)
    return torch.stack(losses).mean()


TASK_VARIANTS = {
    "alpha_only": {"return": 0.0, "alpha": ALPHA_WEIGHT, "vol": 0.0, "downside": 0.0, "regime": 0.0, "rank": RANK_WEIGHT},
    "alpha_vol": {"return": RETURN_WEIGHT, "alpha": ALPHA_WEIGHT, "vol": VOL_WEIGHT, "downside": 0.0, "regime": 0.0, "rank": RANK_WEIGHT},
    "alpha_downside": {"return": RETURN_WEIGHT, "alpha": ALPHA_WEIGHT, "vol": 0.0, "downside": DOWNSIDE_WEIGHT, "regime": 0.0, "rank": RANK_WEIGHT},
    "alpha_vol_regime": {"return": RETURN_WEIGHT, "alpha": ALPHA_WEIGHT, "vol": VOL_WEIGHT, "downside": DOWNSIDE_WEIGHT, "regime": REGIME_WEIGHT, "rank": RANK_WEIGHT},
}
for requested in VARIANT_NAMES:
    if requested not in TASK_VARIANTS:
        raise KeyError(f"Unknown variant '{requested}'. Available: {sorted(TASK_VARIANTS)}")


def batch_loss(
    model: nn.Module,
    xb: torch.Tensor,
    y_return: torch.Tensor,
    y_alpha: torch.Tensor,
    y_vol: torch.Tensor,
    y_downside: torch.Tensor,
    y_regime: torch.Tensor,
    sample_weight: torch.Tensor,
    date_codes: torch.Tensor,
    *,
    variant: dict[str, float],
    denoise: bool,
) -> tuple[torch.Tensor, dict[str, float]]:
    clean_x = xb
    noisy_x = apply_feature_denoising(clean_x, MASK_RATE) if denoise else clean_x
    output = model(noisy_x)
    feature_weights = reconstruction_weight_tensor.to(clean_x.device)
    recon_loss = ((output["reconstruction"] - clean_x).pow(2) * feature_weights).mean()
    return_loss = weighted_mse(output["expected_return"], y_return, sample_weight)
    alpha_loss = weighted_mse(output["expected_alpha"], y_alpha, sample_weight)
    vol_loss = weighted_mse(output["expected_vol_log"], y_vol, sample_weight)
    downside_loss = F.binary_cross_entropy_with_logits(output["downside_logit"], y_downside, weight=sample_weight)
    regime_loss = F.cross_entropy(output["regime_logits"], y_regime, reduction="none")
    regime_loss = (regime_loss * sample_weight).mean()
    rank_loss = pairwise_rank_loss(output["expected_alpha"], y_alpha, date_codes)
    total = (
        RECON_WEIGHT * recon_loss
        + variant["return"] * return_loss
        + variant["alpha"] * alpha_loss
        + variant["vol"] * vol_loss
        + variant["downside"] * downside_loss
        + variant["regime"] * regime_loss
        + variant["rank"] * rank_loss
    )
    parts = {
        "loss": float(total.detach().cpu()),
        "recon": float(recon_loss.detach().cpu()),
        "return": float(return_loss.detach().cpu()),
        "alpha": float(alpha_loss.detach().cpu()),
        "vol": float(vol_loss.detach().cpu()),
        "downside": float(downside_loss.detach().cpu()),
        "regime": float(regime_loss.detach().cpu()),
        "rank": float(rank_loss.detach().cpu()),
    }
    return total, parts


def iter_date_batches(
    X: np.ndarray,
    y_return: np.ndarray,
    y_alpha: np.ndarray,
    y_vol: np.ndarray,
    y_downside: np.ndarray,
    y_regime: np.ndarray,
    sample_weight: np.ndarray,
    dates: np.ndarray,
    *,
    dates_per_batch: int,
    shuffle: bool,
    rng: np.random.Generator,
):
    date_series = pd.Series(pd.to_datetime(dates))
    unique_dates = np.array(pd.unique(date_series))
    if shuffle:
        rng.shuffle(unique_dates)
    for start in range(0, len(unique_dates), dates_per_batch):
        chosen_dates = unique_dates[start : start + dates_per_batch]
        idx = np.flatnonzero(np.isin(date_series.to_numpy(), chosen_dates))
        if shuffle:
            rng.shuffle(idx)
        date_codes = pd.factorize(date_series.iloc[idx])[0].astype(np.int64)
        yield (
            X[idx],
            y_return[idx],
            y_alpha[idx],
            y_vol[idx],
            y_downside[idx],
            y_regime[idx],
            sample_weight[idx],
            date_codes,
        )

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_dates = meta_train["date"].to_numpy(dtype="datetime64[ns]")
val_dates = meta_val["date"].to_numpy(dtype="datetime64[ns]")


def new_model() -> RegimeAwareSupervisedLSTMAutoencoder:
    return RegimeAwareSupervisedLSTMAutoencoder(
        input_dim=len(ALL_FEATURE_NAMES),
        hidden_size=HIDDEN_SIZE,
        latent_dim=LATENT_DIM,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
        regime_classes=len(REGIME_CLASSES),
    ).to(device)


def evaluate_losses(model: nn.Module, variant: dict[str, float]) -> dict[str, float]:
    model.eval()
    rng = np.random.default_rng(RANDOM_SEED)
    keys = ["loss", "recon", "return", "alpha", "vol", "downside", "regime", "rank"]
    totals = {key: 0.0 for key in keys}
    totals["n"] = 0.0
    with torch.no_grad():
        for xb_np, yr_np, ya_np, yv_np, yd_np, yg_np, sw_np, date_codes_np in iter_date_batches(
            X_val_seq,
            y_val_return,
            y_val_alpha,
            y_val_vol,
            y_val_downside,
            y_val_regime,
            w_val,
            val_dates,
            dates_per_batch=DATES_PER_BATCH,
            shuffle=False,
            rng=rng,
        ):
            xb = torch.as_tensor(xb_np, dtype=torch.float32, device=device)
            yr = torch.as_tensor(yr_np, dtype=torch.float32, device=device)
            ya = torch.as_tensor(ya_np, dtype=torch.float32, device=device)
            yv = torch.as_tensor(yv_np, dtype=torch.float32, device=device)
            yd = torch.as_tensor(yd_np, dtype=torch.float32, device=device)
            yg = torch.as_tensor(yg_np, dtype=torch.long, device=device)
            sw = torch.as_tensor(sw_np, dtype=torch.float32, device=device)
            dc = torch.as_tensor(date_codes_np, dtype=torch.long, device=device)
            _, parts = batch_loss(model, xb, yr, ya, yv, yd, yg, sw, dc, variant=variant, denoise=False)
            batch_n = float(len(xb_np))
            for key in keys:
                totals[key] += parts[key] * batch_n
            totals["n"] += batch_n
    return {key: totals[key] / max(totals["n"], 1.0) for key in keys}


def train_variant(variant_name: str) -> dict[str, object]:
    variant = TASK_VARIANTS[variant_name]
    torch.manual_seed(RANDOM_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_SEED)
    model_obj = new_model()
    optimizer = torch.optim.AdamW(model_obj.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    history_rows = []
    best_state = {key: value.detach().cpu().clone() for key, value in model_obj.state_dict().items()}
    best_val_loss = float("inf")
    patience_counter = 0
    rng = np.random.default_rng(RANDOM_SEED)

    for epoch in range(1, EPOCHS + 1):
        model_obj.train()
        keys = ["loss", "recon", "return", "alpha", "vol", "downside", "regime", "rank"]
        train_totals = {key: 0.0 for key in keys}
        train_totals["n"] = 0.0
        for xb_np, yr_np, ya_np, yv_np, yd_np, yg_np, sw_np, date_codes_np in iter_date_batches(
            X_train_seq,
            y_train_return,
            y_train_alpha,
            y_train_vol,
            y_train_downside,
            y_train_regime,
            w_train,
            train_dates,
            dates_per_batch=DATES_PER_BATCH,
            shuffle=True,
            rng=rng,
        ):
            xb = torch.as_tensor(xb_np, dtype=torch.float32, device=device)
            yr = torch.as_tensor(yr_np, dtype=torch.float32, device=device)
            ya = torch.as_tensor(ya_np, dtype=torch.float32, device=device)
            yv = torch.as_tensor(yv_np, dtype=torch.float32, device=device)
            yd = torch.as_tensor(yd_np, dtype=torch.float32, device=device)
            yg = torch.as_tensor(yg_np, dtype=torch.long, device=device)
            sw = torch.as_tensor(sw_np, dtype=torch.float32, device=device)
            dc = torch.as_tensor(date_codes_np, dtype=torch.long, device=device)

            optimizer.zero_grad(set_to_none=True)
            loss, parts = batch_loss(model_obj, xb, yr, ya, yv, yd, yg, sw, dc, variant=variant, denoise=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model_obj.parameters(), GRAD_CLIP)
            optimizer.step()

            batch_n = float(len(xb_np))
            for key in keys:
                train_totals[key] += parts[key] * batch_n
            train_totals["n"] += batch_n

        train_metrics = {key: train_totals[key] / max(train_totals["n"], 1.0) for key in keys}
        val_metrics = evaluate_losses(model_obj, variant)
        row = {"variant": variant_name, "epoch": epoch}
        row.update({f"train_{k}": v for k, v in train_metrics.items()})
        row.update({f"val_{k}": v for k, v in val_metrics.items()})
        history_rows.append(row)
        print(
            f"{variant_name} epoch {epoch:02d}/{EPOCHS} | train {train_metrics['loss']:.4f} | "
            f"val {val_metrics['loss']:.4f} | alpha {val_metrics['alpha']:.4f} | "
            f"downside {val_metrics['downside']:.4f} | regime {val_metrics['regime']:.4f}"
        )

        if val_metrics["loss"] < best_val_loss - 1e-5:
            best_val_loss = val_metrics["loss"]
            best_state = {key: value.detach().cpu().clone() for key, value in model_obj.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"Early stopping {variant_name} at epoch {epoch}.")
                break

    model_obj.load_state_dict(best_state)
    return {
        "name": variant_name,
        "model": model_obj,
        "history": pd.DataFrame(history_rows),
        "best_val_loss": best_val_loss,
        "loss_weights": variant,
    }


trained_variants = {}
for variant_name in VARIANT_NAMES:
    trained_variants[variant_name] = train_variant(variant_name)

history = pd.concat([entry["history"] for entry in trained_variants.values()], ignore_index=True)
display(history.groupby("variant").tail(1))

alpha_only epoch 01/18 | train 3.2980 | val 5.4026 | alpha 1.5720 | downside 0.7577 | regime 1.7187
alpha_only epoch 02/18 | train 3.0336 | val 5.2648 | alpha 1.5798 | downside 0.7641 | regime 1.7082
alpha_only epoch 03/18 | train 2.9049 | val 5.2686 | alpha 1.6198 | downside 0.7636 | regime 1.7247
alpha_only epoch 04/18 | train 2.7821 | val 5.3332 | alpha 1.6773 | downside 0.7680 | regime 1.7158
alpha_only epoch 05/18 | train 2.6573 | val 5.3780 | alpha 1.7033 | downside 0.7700 | regime 1.7223
alpha_only epoch 06/18 | train 2.5577 | val 5.4856 | alpha 1.7661 | downside 0.7691 | regime 1.7292
alpha_only epoch 07/18 | train 2.4741 | val 5.5599 | alpha 1.8150 | downside 0.7667 | regime 1.7199
Early stopping alpha_only at epoch 7.
alpha_vol epoch 01/18 | train 3.9187 | val 6.5045 | alpha 1.5728 | downside 0.7489 | regime 1.7332
alpha_vol epoch 02/18 | train 3.6277 | val 6.3638 | alpha 1.5722 | downside 0.7610 | regime 1.7060
alpha_vol epoch 03/18 | train 3.4799 | val 6.3193 | alpha 1.6165

,variant,epoch,train_loss,train_recon,train_return,train_alpha,train_vol,train_downside,train_regime,train_rank,val_loss,val_recon,val_return,val_alpha,val_vol,val_downside,val_regime,val_rank
6,alpha_only,7,2.474115,0.472227,1.154738,0.902877,1.172024,0.723963,1.524093,0.653779,5.559868,1.713809,1.902494,1.814966,2.355204,0.766650,1.719903,0.720422
14,alpha_vol,8,2.839134,0.470266,0.875757,0.846899,0.749232,0.711878,1.523174,0.646332,6.923181,1.692858,2.368596,1.917457,1.665087,0.762637,1.727789,0.734930
22,alpha_downside,8,2.897781,0.473540,0.883042,0.848621,1.163780,0.521873,1.512486,0.643715,6.787969,1.705713,2.358226,1.946809,2.301459,0.630455,1.716845,0.736029
31,alpha_vol_regime,9,3.185748,0.472413,0.859260,0.828645,0.733959,0.521879,0.262083,0.641974,7.734828,1.773390,2.227716,1.924853,1.987154,0.624904,0.885143,0.729383


## Validation Diagnostics And Variant Selection

In [10]:
def predict_scaled(model: nn.Module, X: np.ndarray, batch_size: int = 4096) -> pd.DataFrame:
    model.eval()
    returns, alphas, vols, downside_probs, latent_norms, regime_probs = [], [], [], [], [], []
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            xb = torch.as_tensor(X[start : start + batch_size], dtype=torch.float32, device=device)
            output = model(xb)
            returns.append(output["expected_return"].detach().cpu().numpy())
            alphas.append(output["expected_alpha"].detach().cpu().numpy())
            vols.append(output["expected_vol_log"].detach().cpu().numpy())
            downside_probs.append(torch.sigmoid(output["downside_logit"]).detach().cpu().numpy())
            regime_probs.append(torch.softmax(output["regime_logits"], dim=1).detach().cpu().numpy())
            latent_norms.append(torch.linalg.norm(output["latent"], dim=1).detach().cpu().numpy())
    result = pd.DataFrame(
        {
            "return_scaled_pred": np.concatenate(returns),
            "alpha_scaled_pred": np.concatenate(alphas),
            "vol_scaled_pred": np.concatenate(vols),
            "expected_downside_risk": np.concatenate(downside_probs),
            "latent_norm": np.concatenate(latent_norms),
        }
    )
    regime_array = np.concatenate(regime_probs)
    for idx, name in enumerate(REGIME_CLASSES):
        result[f"regime_prob_{name}"] = regime_array[:, idx]
    return result


def unscale_predictions(predicted: pd.DataFrame) -> pd.DataFrame:
    result = predicted.copy()
    result["expected_return"] = result["return_scaled_pred"] * target_stds["return"] + target_means["return"]
    result["expected_alpha"] = result["alpha_scaled_pred"] * target_stds["alpha"] + target_means["alpha"]
    vol_log = result["vol_scaled_pred"] * target_stds["vol_log"] + target_means["vol_log"]
    result["expected_forward_volatility"] = np.expm1(vol_log).clip(lower=1e-4)
    result["regime_confidence"] = (
        result["regime_prob_calm_risk_on"]
        + 0.50 * result["regime_prob_rebound"]
        - 0.50 * result["regime_prob_high_vol"]
        - result["regime_prob_drawdown"]
    )
    return result


def add_scores(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    result["expected_volatility"] = np.clip(result["historical_vol_20d"].to_numpy(dtype=float), 1e-4, None)
    result["base_alpha_vol_score"] = result["expected_alpha"] / result["expected_volatility"]
    result["blended_regime_score"] = (
        result["base_alpha_vol_score"]
        - DOWNSIDE_SCORE_PENALTY * result["expected_downside_risk"]
        - VOL_SCORE_PENALTY * result["expected_forward_volatility"]
        + REGIME_SCORE_BONUS * result["regime_confidence"]
    )
    result["uncertainty"] = np.abs(result["expected_return"] - result["expected_alpha"])
    return result


def rank_normalize_by_date(frame: pd.DataFrame, score_col: str, out_col: str) -> pd.DataFrame:
    result = frame.copy()
    pct_rank = result.groupby("date")[score_col].rank(method="average", pct=True)
    result[out_col] = 2.0 * pct_rank - 1.0
    return result


def daily_rank_ic(frame: pd.DataFrame, score_col: str, target_col: str) -> pd.DataFrame:
    rows = []
    for date_value, group in frame.groupby("date", sort=True):
        if group[score_col].nunique() < 2 or group[target_col].nunique() < 2:
            continue
        rows.append({"date": pd.Timestamp(date_value), "rank_ic": group[score_col].rank().corr(group[target_col].rank())})
    return pd.DataFrame(rows)


def top_bottom_spread(frame: pd.DataFrame, score_col: str, target_col: str, buckets: int = 5) -> float:
    rows = []
    for _, group in frame.groupby("date", sort=True):
        group = group.sort_values(score_col, ascending=False).reset_index(drop=True)
        if len(group) < buckets:
            continue
        bucket_size = max(1, len(group) // buckets)
        rows.append(float(group.head(bucket_size)[target_col].mean() - group.tail(bucket_size)[target_col].mean()))
    return float(np.nanmean(rows)) if rows else np.nan


def score_diagnostics(frame: pd.DataFrame, score_col: str) -> dict[str, float]:
    high_risk = frame["market_regime"].isin(["high_vol", "drawdown"])
    rank_ic = daily_rank_ic(frame, score_col, "target_alpha")
    high_risk_rank_ic = daily_rank_ic(frame.loc[high_risk], score_col, "target_alpha")
    spread_all = top_bottom_spread(frame, score_col, "target_alpha")
    spread_high_risk = top_bottom_spread(frame.loc[high_risk], score_col, "target_alpha")
    robust_score = (
        100.0 * spread_all
        + 75.0 * np.nan_to_num(spread_high_risk, nan=0.0)
        + 0.25 * np.nan_to_num(rank_ic["rank_ic"].mean() if not rank_ic.empty else np.nan, nan=0.0)
        + 0.25 * np.nan_to_num(high_risk_rank_ic["rank_ic"].mean() if not high_risk_rank_ic.empty else np.nan, nan=0.0)
        - 50.0 * max(0.0, -np.nan_to_num(spread_high_risk, nan=0.0))
    )
    return {
        "rank_ic": float(rank_ic["rank_ic"].mean()) if not rank_ic.empty else np.nan,
        "rank_ic_high_risk": float(high_risk_rank_ic["rank_ic"].mean()) if not high_risk_rank_ic.empty else np.nan,
        "top_bottom_alpha_spread": spread_all,
        "top_bottom_alpha_spread_high_risk": spread_high_risk,
        "robust_validation_score": float(robust_score),
    }


def diagnostics_by_year(frame: pd.DataFrame, score_col: str) -> pd.DataFrame:
    rank_ic = daily_rank_ic(frame, score_col, "target_alpha")
    if rank_ic.empty:
        return pd.DataFrame(columns=["year", "mean_rank_ic", "observations"])
    rank_ic["year"] = rank_ic["date"].dt.year
    return rank_ic.groupby("year").agg(mean_rank_ic=("rank_ic", "mean"), observations=("rank_ic", "count")).reset_index()


def diagnostics_by_regime(frame: pd.DataFrame, score_col: str) -> pd.DataFrame:
    rows = []
    for regime_name, group in frame.groupby("market_regime", sort=True):
        metrics = score_diagnostics(group, score_col)
        rows.append({"market_regime": regime_name, "rows": len(group), **metrics})
    return pd.DataFrame(rows)


variant_rows = []
variant_diagnostics = {}
for variant_name, entry in trained_variants.items():
    val_pred = unscale_predictions(predict_scaled(entry["model"], X_val_seq))
    diag = add_scores(pd.concat([meta_val.copy(), val_pred], axis=1))
    diag = rank_normalize_by_date(diag, "base_alpha_vol_score", "base_rank_score")
    diag = rank_normalize_by_date(diag, "blended_regime_score", "blended_rank_score")
    variant_diagnostics[variant_name] = diag
    for score_col in ["base_alpha_vol_score", "blended_regime_score"]:
        metrics = score_diagnostics(diag, score_col)
        variant_rows.append({"variant": variant_name, "score_col": score_col, **metrics})

variant_comparison = pd.DataFrame(variant_rows).sort_values("robust_validation_score", ascending=False).reset_index(drop=True)
display(variant_comparison)

if SELECT_BEST_VARIANT:
    selected_variant_name = str(variant_comparison.iloc[0]["variant"])
elif FINAL_VARIANT_NAME in trained_variants:
    selected_variant_name = FINAL_VARIANT_NAME
else:
    selected_variant_name = next(iter(trained_variants))

model = trained_variants[selected_variant_name]["model"]
selected_val_diagnostics = variant_diagnostics[selected_variant_name]
selected_score_col = "blended_regime_score" if selected_variant_name == "alpha_vol_regime" else "base_alpha_vol_score"
val_return_mse = float(np.mean((selected_val_diagnostics["expected_return"] - selected_val_diagnostics["target_return"]) ** 2))
val_alpha_mse = float(np.mean((selected_val_diagnostics["expected_alpha"] - selected_val_diagnostics["target_alpha"]) ** 2))
val_downside_brier = float(np.mean((selected_val_diagnostics["expected_downside_risk"] - selected_val_diagnostics["target_bottom_quintile"]) ** 2))
val_regime_accuracy = float((selected_val_diagnostics[[f"regime_prob_{name}" for name in REGIME_CLASSES]].to_numpy().argmax(axis=1) == selected_val_diagnostics["market_regime_code"].to_numpy()).mean())

print("Selected variant:", selected_variant_name)
print("Validation return MSE:", val_return_mse)
print("Validation alpha MSE:", val_alpha_mse)
print("Validation downside Brier:", val_downside_brier)
print("Validation regime accuracy:", val_regime_accuracy)
print("Rank IC by year using blended score:")
display(diagnostics_by_year(selected_val_diagnostics, "blended_regime_score"))
print("Regime diagnostics using blended score:")
display(diagnostics_by_regime(selected_val_diagnostics, "blended_regime_score"))

,variant,score_col,rank_ic,rank_ic_high_risk,top_bottom_alpha_spread,top_bottom_alpha_spread_high_risk,robust_validation_score
0,alpha_only,base_alpha_vol_score,0.029388,0.090812,0.003766,0.012733,1.361621
1,alpha_vol,base_alpha_vol_score,0.013212,0.038095,0.004182,0.010995,1.255678
2,alpha_downside,base_alpha_vol_score,0.015484,0.041107,0.002809,0.008498,0.932401
3,alpha_downside,blended_regime_score,0.015723,0.042423,0.001930,0.007049,0.736188
4,alpha_vol_regime,base_alpha_vol_score,0.023555,0.052714,0.002254,0.006137,0.704738
5,alpha_only,blended_regime_score,0.020716,0.057678,0.000706,0.004940,0.460757
6,alpha_vol,blended_regime_score,-0.001115,0.006330,0.000695,0.003840,0.358811
7,alpha_vol_regime,blended_regime_score,0.014725,0.019978,-0.001241,0.000651,-0.066548


Selected variant: alpha_vol_regime
Validation return MSE: 0.0032260038991118777
Validation alpha MSE: 0.0020638153106773945
Validation downside Brier: 0.15784031200096002
Validation regime accuracy: 0.8093465346534654
Rank IC by year using blended score:


,year,mean_rank_ic,observations
0,2020,0.007337,253
1,2021,0.022143,252


Regime diagnostics using blended score:


,market_regime,rows,rank_ic,rank_ic_high_risk,top_bottom_alpha_spread,top_bottom_alpha_spread_high_risk,robust_validation_score
0,calm_risk_on,6475,0.008871,NaN,-0.003499,NaN,-0.347654
1,drawdown,1550,0.043486,0.043486,0.004070,0.004070,0.734078
2,high_vol,3800,0.010390,0.010390,-0.000744,-0.000744,-0.162115
3,rebound,800,0.026971,NaN,0.004385,NaN,0.445266


## Predictions, Rebalance Experiments, And Regime-Aware Portfolio

In [11]:
def select_rebalance_dates(dates: pd.Series | pd.DatetimeIndex, frequency: str = "weekly") -> pd.DatetimeIndex:
    unique_dates = pd.DatetimeIndex(pd.to_datetime(pd.Series(dates).drop_duplicates()).sort_values())
    if frequency == "daily":
        return unique_dates
    date_series = pd.Series(unique_dates, index=unique_dates)
    if frequency == "weekly":
        return pd.DatetimeIndex(date_series.groupby(date_series.index.to_period("W-FRI")).max().to_numpy())
    if frequency == "monthly":
        return pd.DatetimeIndex(date_series.groupby(date_series.index.to_period("M")).max().to_numpy())
    if frequency == "every_5_trading_days":
        return unique_dates[::5]
    if frequency == "every_10_trading_days":
        return unique_dates[::10]
    raise ValueError("frequency must be one of: daily, weekly, monthly, every_5_trading_days, every_10_trading_days")


def apply_weight_cap(raw_weights: pd.Series, max_weight: float) -> pd.Series:
    raw = raw_weights.clip(lower=0.0).astype(float)
    raw = raw.loc[raw > 0.0]
    if raw.empty:
        return raw_weights * 0.0
    feasible_cap = max(float(max_weight), 1.0 / len(raw) + 1e-8)
    weights = pd.Series(0.0, index=raw.index, dtype=float)
    remaining = raw.copy()
    budget = 1.0
    while not remaining.empty:
        scaled = remaining / remaining.sum() * budget
        over_cap = scaled > feasible_cap
        if not over_cap.any():
            weights.loc[remaining.index] = scaled
            break
        capped_names = scaled.loc[over_cap].index
        weights.loc[capped_names] = feasible_cap
        budget -= feasible_cap * len(capped_names)
        remaining = remaining.drop(index=capped_names)
        if budget <= 1e-12:
            break
    total = weights.sum()
    if total <= 0.0:
        weights = pd.Series(1.0 / len(raw), index=raw.index, dtype=float)
    elif abs(total - 1.0) > 1e-10:
        weights = weights / total
    return weights


def _price_wide_for_builder(prices: pd.DataFrame, tickers: list[str]) -> pd.DataFrame:
    wide = prices.copy()
    wide["date"] = pd.to_datetime(wide["date"], utc=True).dt.tz_localize(None)
    wide["ticker"] = wide["ticker"].astype(str).str.upper()
    return wide.pivot(index="date", columns="ticker", values="adj_close").sort_index().loc[:, tickers]


def _risk_multiplier(group: pd.DataFrame, proxy_drawdown: float, dynamic_risk: bool) -> float:
    if not dynamic_risk:
        return 1.0
    vol = float(group["spy_vol_20d_ann"].median())
    drawdown = float(group["spy_drawdown_60d"].median())
    p_drawdown = float(group.get("regime_prob_drawdown", pd.Series(0.0, index=group.index)).mean())
    p_high_vol = float(group.get("regime_prob_high_vol", pd.Series(0.0, index=group.index)).mean())
    multiplier = 1.0
    if vol >= regime_thresholds["high_vol_spy_vol_20d_ann"]:
        multiplier -= 0.12
    if drawdown <= regime_thresholds["drawdown_spy_drawdown_60d"]:
        multiplier -= 0.16
    if p_high_vol >= 0.35:
        multiplier -= 0.08
    if p_drawdown >= 0.30:
        multiplier -= 0.10
    if proxy_drawdown <= -0.10:
        multiplier -= 0.10
    if proxy_drawdown <= -0.20:
        multiplier -= 0.10
    return float(np.clip(multiplier, 0.55, 1.05))


def weights_from_regime_scores(
    predictions: pd.DataFrame,
    *,
    score_col: str,
    dataset_name: str,
    strategy_name: str,
    max_weight: float,
    top_fraction: float,
    turnover_blend: float,
    dynamic_risk: bool,
    beta_target: float | None,
    prices: pd.DataFrame,
    universe_tickers: list[str] | None = None,
) -> tuple[PortfolioWeights, pd.DataFrame]:
    tickers = sorted(predictions["ticker"].unique()) if universe_tickers is None else list(universe_tickers)
    tickers = [ticker for ticker in tickers if ticker in set(predictions["ticker"])]
    price_wide = _price_wide_for_builder(prices, tickers)
    rows = []
    risk_rows = []
    previous = None
    previous_date = None
    proxy_nav = 1.0
    proxy_peak = 1.0

    for date_value, group in predictions.groupby("date", sort=True):
        date_value = pd.Timestamp(date_value)
        if previous is not None and previous_date is not None:
            if previous_date in price_wide.index and date_value in price_wide.index:
                period_returns = price_wide.loc[date_value, tickers] / price_wide.loc[previous_date, tickers] - 1.0
                period_returns = period_returns.replace([np.inf, -np.inf], np.nan).fillna(0.0)
                proxy_nav *= float(1.0 + (previous * period_returns).sum())
                proxy_peak = max(proxy_peak, proxy_nav)
        proxy_drawdown = proxy_nav / max(proxy_peak, 1e-12) - 1.0
        multiplier = _risk_multiplier(group, proxy_drawdown, dynamic_risk)
        effective_max_weight = max(DYNAMIC_MIN_MAX_WEIGHT, max_weight * multiplier, 1.0 / max(len(tickers), 1) + 1e-6)
        effective_top_fraction = min(0.90, top_fraction + (1.0 - multiplier) * 0.45)
        score_dispersion = float(group[score_col].quantile(0.80) - group[score_col].quantile(0.20))
        if dynamic_risk and multiplier > 0.95 and score_dispersion > 0.50:
            effective_top_fraction = max(0.30, effective_top_fraction - 0.10)

        group = group.sort_values(score_col, ascending=False).reset_index(drop=True)
        min_names = min(len(group), int(math.ceil(1.0 / effective_max_weight)))
        top_n = min(len(group), max(min_names, int(math.ceil(len(group) * effective_top_fraction))))
        selected = group.head(top_n).copy()
        beta_penalty = 1.0 + BETA_SCORE_PENALTY * selected.get("beta_60d_spy", pd.Series(1.0, index=selected.index)).abs().fillna(1.0)
        raw_scores = (selected[score_col] - selected[score_col].min() + 1e-6) / beta_penalty
        raw = pd.Series(raw_scores.to_numpy(dtype=float), index=selected["ticker"])
        raw = raw.loc[raw.index.isin(tickers)]
        capped = apply_weight_cap(raw, max_weight=effective_max_weight)
        row = pd.Series(0.0, index=tickers, dtype=float)
        row.loc[capped.index] = capped.to_numpy(dtype=float)
        if row.sum() <= 0.0:
            fallback = [ticker for ticker in group["ticker"] if ticker in tickers]
            if not fallback:
                fallback = tickers
            row.loc[fallback] = 1.0 / len(fallback)

        betas = group.drop_duplicates("ticker").set_index("ticker").reindex(tickers)["beta_60d_spy"].astype(float).fillna(1.0)
        portfolio_beta = float((row * betas).sum())
        beta_blend = 0.0
        if beta_target is not None and portfolio_beta > beta_target:
            equal_row = pd.Series(1.0 / len(tickers), index=tickers, dtype=float)
            equal_beta = float((equal_row * betas).sum())
            if portfolio_beta > equal_beta:
                beta_blend = float(np.clip((portfolio_beta - beta_target) / (portfolio_beta - equal_beta), 0.0, 0.75))
                row = (1.0 - beta_blend) * row + beta_blend * equal_row
                row = row / row.sum()
                portfolio_beta = float((row * betas).sum())

        if previous is not None:
            row = turnover_blend * row + (1.0 - turnover_blend) * previous
            row = row / row.sum()
        row.name = date_value
        rows.append(row)
        risk_rows.append(
            {
                "date": date_value,
                "risk_multiplier": multiplier,
                "effective_max_weight": effective_max_weight,
                "effective_top_fraction": effective_top_fraction,
                "proxy_drawdown": proxy_drawdown,
                "portfolio_beta_estimate": portfolio_beta,
                "beta_blend": beta_blend,
                "active_names": int((row > 1e-8).sum()),
                "realized_max_weight": float(row.max()),
            }
        )
        previous = row
        previous_date = date_value

    weights = pd.DataFrame(rows)
    weights.index.name = "date"
    metadata = {
        "score_col": score_col,
        "rebalance_frequency": REBALANCE_FREQUENCY,
        "base_max_weight": max_weight,
        "base_top_fraction": top_fraction,
        "turnover_blend": turnover_blend,
        "dynamic_risk": dynamic_risk,
        "beta_target": beta_target,
        "universe_ticker_count": len(tickers),
    }
    return PortfolioWeights(weights=weights, dataset_name=dataset_name, strategy_name=strategy_name, metadata=metadata), pd.DataFrame(risk_rows)


test_pred_scaled = unscale_predictions(predict_scaled(model, X_test_seq_all))
test_scored_all = add_scores(pd.concat([meta_test_all.copy(), test_pred_scaled], axis=1))
test_scored_all = rank_normalize_by_date(test_scored_all, "base_alpha_vol_score", "base_rank_score")
test_scored_all = rank_normalize_by_date(test_scored_all, "blended_regime_score", "blended_rank_score")

candidate_portfolios = {}
candidate_risk_logs = {}
for frequency in ["weekly", "every_5_trading_days", "every_10_trading_days"]:
    rebalance_dates = select_rebalance_dates(test_scored_all["date"], frequency)
    scored = test_scored_all.loc[test_scored_all["date"].isin(rebalance_dates)].copy()
    candidate_predictions = scored.loc[
        :,
        [
            "date",
            "ticker",
            "expected_return",
            "expected_alpha",
            "expected_volatility",
            "expected_forward_volatility",
            "expected_downside_risk",
            "uncertainty",
            "base_alpha_vol_score",
            "blended_regime_score",
            "base_rank_score",
            "blended_rank_score",
            "regime_confidence",
            "regime_prob_calm_risk_on",
            "regime_prob_high_vol",
            "regime_prob_drawdown",
            "regime_prob_rebound",
            "spy_vol_20d_ann",
            "spy_drawdown_60d",
            "spy_momentum_20d",
            "beta_60d_spy",
        ],
    ].copy()
    candidate_predictions["horizon"] = HORIZON
    candidate_predictions = candidate_predictions.loc[:, ["date", "ticker", "horizon"] + [c for c in candidate_predictions.columns if c not in {"date", "ticker", "horizon"}]]
    candidate_predictions = validate_prediction_frame(candidate_predictions, dataset_name=DATASET_NAME, horizon=HORIZON, repo_root=repo_root)

    if frequency == "weekly":
        base_portfolio, base_risk = weights_from_regime_scores(
            candidate_predictions,
            score_col="base_rank_score",
            dataset_name=DATASET_NAME,
            strategy_name=f"{MODEL_NAME}_weekly_base_score",
            max_weight=BASE_MAX_WEIGHT,
            top_fraction=BASE_TOP_FRACTION,
            turnover_blend=TURNOVER_BLEND,
            dynamic_risk=False,
            beta_target=None,
            prices=prices,
            universe_tickers=tradable_tickers,
        )
        candidate_portfolios["weekly_base_score"] = base_portfolio
        candidate_risk_logs["weekly_base_score"] = base_risk

    dynamic = frequency == "weekly"
    name = f"{frequency}_regime_dynamic" if dynamic else f"{frequency}_regime_static"
    portfolio_obj, risk_log = weights_from_regime_scores(
        candidate_predictions,
        score_col="blended_rank_score",
        dataset_name=DATASET_NAME,
        strategy_name=f"{MODEL_NAME}_{name}",
        max_weight=BASE_MAX_WEIGHT,
        top_fraction=BASE_TOP_FRACTION,
        turnover_blend=TURNOVER_BLEND,
        dynamic_risk=dynamic,
        beta_target=BETA_TARGET if dynamic else None,
        prices=prices,
        universe_tickers=tradable_tickers,
    )
    candidate_portfolios[name] = portfolio_obj
    candidate_risk_logs[name] = risk_log

portfolio = candidate_portfolios["weekly_regime_dynamic"]
risk_log = candidate_risk_logs["weekly_regime_dynamic"]
validated_weights = validate_weights_frame(portfolio.weights, dataset_name=DATASET_NAME, repo_root=repo_root)
weekly_dates = pd.DatetimeIndex(validated_weights.index)
predictions = test_scored_all.loc[test_scored_all["date"].isin(weekly_dates)].copy()
predictions = predictions.loc[
    :,
    [
        "date",
        "ticker",
        "expected_return",
        "expected_alpha",
        "expected_volatility",
        "expected_forward_volatility",
        "expected_downside_risk",
        "uncertainty",
        "base_alpha_vol_score",
        "blended_regime_score",
        "base_rank_score",
        "blended_rank_score",
        "regime_confidence",
        "regime_prob_calm_risk_on",
        "regime_prob_high_vol",
        "regime_prob_drawdown",
        "regime_prob_rebound",
        "spy_vol_20d_ann",
        "spy_drawdown_60d",
        "spy_momentum_20d",
        "beta_60d_spy",
    ],
].copy()
predictions["horizon"] = HORIZON
predictions = predictions.loc[:, ["date", "ticker", "horizon"] + [c for c in predictions.columns if c not in {"date", "ticker", "horizon"}]]
predictions = validate_prediction_frame(predictions, dataset_name=DATASET_NAME, horizon=HORIZON, repo_root=repo_root)

print("Predictions:", predictions.shape)
print("Weekly weight rows:", validated_weights.shape)
print("Prediction date range:", predictions["date"].min(), "->", predictions["date"].max())
print("Average active names:", float((validated_weights > 1e-8).sum(axis=1).mean()))
print("Average realized max weight:", float(risk_log["realized_max_weight"].mean()))
display(predictions.head())
display(risk_log.head())
display(validated_weights.head())

Predictions: (5198, 22)
Weekly weight rows: (208, 25)
Prediction date range: 2022-01-07 00:00:00 -> 2025-12-23 00:00:00
Average active names: 24.79326923076923
Average realized max weight: 0.07837666880744934


,date,ticker,horizon,expected_return,expected_alpha,expected_volatility,expected_forward_volatility,expected_downside_risk,uncertainty,base_alpha_vol_score,...,blended_rank_score,regime_confidence,regime_prob_calm_risk_on,regime_prob_high_vol,regime_prob_drawdown,regime_prob_rebound,spy_vol_20d_ann,spy_drawdown_60d,spy_momentum_20d,beta_60d_spy
0,2022-01-07,AAPL,5,0.010556,0.009401,0.291344,0.223164,0.110177,0.001155,0.032267,...,0.20,-0.361062,0.122392,0.705070,0.144792,0.027746,0.152736,-0.024324,0.002954,1.164403
1,2022-01-07,ADBE,5,0.029642,0.021665,0.519157,0.313350,0.148287,0.007977,0.041730,...,0.04,-0.327550,0.174474,0.492132,0.281770,0.051625,0.152736,-0.024324,0.002954,2.093537
2,2022-01-07,ADI,5,0.010750,0.004796,0.242832,0.267945,0.138554,0.005954,0.019752,...,-0.04,-0.234635,0.218272,0.591196,0.168383,0.022148,0.152736,-0.024324,0.002954,1.297958
3,2022-01-07,AMAT,5,0.007514,0.002413,0.408457,0.324751,0.172684,0.005101,0.005907,...,0.68,0.370105,0.593209,0.324889,0.067740,0.014161,0.152736,-0.024324,0.002954,1.853233
4,2022-01-07,AMD,5,0.022734,0.020026,0.588302,0.378952,0.184027,0.002708,0.034041,...,0.44,0.144809,0.459627,0.314937,0.180045,0.045391,0.152736,-0.024324,0.002954,2.407294


,date,risk_multiplier,effective_max_weight,effective_top_fraction,proxy_drawdown,portfolio_beta_estimate,beta_blend,active_names,realized_max_weight
0,2022-01-07,0.80,0.080,0.5900,0.000000,1.559718,0.75,25,0.050000
1,2022-01-14,0.92,0.092,0.5360,0.000000,1.608107,0.75,25,0.051500
2,2022-01-21,0.62,0.062,0.6710,-0.099526,1.649266,0.75,25,0.048500
3,2022-01-28,0.55,0.055,0.7025,-0.104245,1.650516,0.75,25,0.046125
4,2022-02-04,0.62,0.062,0.6710,-0.081277,1.627468,0.00,25,0.054062


,AAPL,MSFT,NVDA,AMZN,GOOGL,META,AVGO,NFLX,TSLA,ADBE,...,INTC,MU,AMAT,LRCX,KLAC,PANW,NOW,ADI,INTU,CSCO
date,,,,,,,,,,,,,,,,,,,,,
2022-01-07,0.050000,0.050000,0.050000,0.045710,0.030,0.050000,0.030000,0.050000,0.030000,0.039220,...,0.030,0.050000,0.050000,0.050000,0.050000,0.030,0.030000,0.035070,0.030000,0.030
2022-01-14,0.051500,0.051500,0.051500,0.037855,0.030,0.051500,0.030000,0.040000,0.034029,0.046110,...,0.030,0.040000,0.040000,0.051500,0.042180,0.030,0.030000,0.041822,0.040861,0.030
2022-01-21,0.048500,0.048500,0.048500,0.041678,0.030,0.048500,0.037750,0.035000,0.032015,0.045805,...,0.030,0.035000,0.036000,0.048500,0.043840,0.030,0.037750,0.043661,0.043180,0.030
2022-01-28,0.046125,0.046125,0.046125,0.042714,0.030,0.046125,0.033875,0.039375,0.037882,0.044777,...,0.030,0.039375,0.039875,0.046125,0.043795,0.030,0.040750,0.043706,0.043465,0.030
2022-02-04,0.054062,0.054062,0.023062,0.052357,0.015,0.054062,0.047938,0.019688,0.049941,0.022389,...,0.046,0.023688,0.050937,0.054062,0.052898,0.015,0.020375,0.052853,0.052733,0.046


## Backtests, Regime Evaluation, And Stress Tables

In [12]:
def write_artifacts_for_notebook(result, output_dir: Path) -> dict[str, str]:
    if os.environ.get("SKIP_QUANTSTATS", "0") != "1":
        return write_backtest_artifacts(result, output_dir)

    output_path = Path(output_dir).resolve()
    output_path.mkdir(parents=True, exist_ok=True)
    paths = {
        "weights": output_path / "weights.parquet",
        "nav": output_path / "nav.parquet",
        "returns": output_path / "returns.parquet",
        "turnover": output_path / "turnover.parquet",
        "benchmarks": output_path / "benchmarks.parquet",
        "metrics": output_path / "metrics.json",
        "metrics_table": output_path / "metrics_table.parquet",
        "quantstats_report": output_path / "quantstats.html",
    }
    result.weights.to_parquet(paths["weights"])
    result.nav.to_frame(name="nav").to_parquet(paths["nav"])
    result.returns.to_frame(name="returns").to_parquet(paths["returns"])
    result.turnover.to_frame(name="turnover").to_parquet(paths["turnover"])
    result.benchmark_returns.to_parquet(paths["benchmarks"])
    paths["metrics"].write_text(json.dumps(result.metrics, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    pd.DataFrame([{"metric": key, "value": value} for key, value in sorted(result.metrics.items())]).to_parquet(
        paths["metrics_table"], index=False
    )
    paths["quantstats_report"].write_text(
        "<html><body><p>QuantStats skipped because SKIP_QUANTSTATS=1.</p></body></html>\n",
        encoding="utf-8",
    )
    artifact_paths = {key: str(value) for key, value in paths.items()}
    result.artifact_paths.update(artifact_paths)
    return artifact_paths


def stress_table(result, benchmark_name: str) -> pd.DataFrame:
    returns = result.returns.astype(float)
    benchmark_returns = result.benchmark_returns.iloc[:, 0].astype(float)
    if benchmark_name in result.benchmark_returns.columns:
        benchmark_returns = result.benchmark_returns[benchmark_name].astype(float)
    common = pd.concat([returns.rename("strategy"), benchmark_returns.rename("benchmark")], axis=1).dropna()
    rolling_5d = (1.0 + common["strategy"]).rolling(5).apply(np.prod, raw=True) - 1.0
    monthly = (1.0 + common["strategy"]).resample("M").prod() - 1.0
    crash_threshold = common["benchmark"].quantile(0.10)
    rally_threshold = common["benchmark"].quantile(0.90)
    return pd.DataFrame(
        [
            {"stress_metric": "worst_day", "value": float(common["strategy"].min())},
            {"stress_metric": "worst_5_day_period", "value": float(rolling_5d.min())},
            {"stress_metric": "worst_month", "value": float(monthly.min())},
            {"stress_metric": "avg_return_benchmark_worst_decile_days", "value": float(common.loc[common["benchmark"] <= crash_threshold, "strategy"].mean())},
            {"stress_metric": "avg_return_benchmark_rally_days", "value": float(common.loc[common["benchmark"] >= rally_threshold, "strategy"].mean())},
            {"stress_metric": "benchmark_worst_decile_threshold", "value": float(crash_threshold)},
        ]
    )


def regime_return_table(result, context_frame: pd.DataFrame, benchmark_name: str) -> pd.DataFrame:
    returns = result.returns.rename("strategy_return").to_frame()
    benchmark_returns = result.benchmark_returns.iloc[:, 0].rename("benchmark_return")
    if benchmark_name in result.benchmark_returns.columns:
        benchmark_returns = result.benchmark_returns[benchmark_name].rename("benchmark_return")
    context = context_frame.loc[:, ["date", "spy_vol_20d_ann", "spy_drawdown_60d", "market_regime"]].drop_duplicates("date").copy()
    context["date"] = pd.to_datetime(context["date"])
    frame = returns.join(benchmark_returns, how="left").reset_index()
    frame = frame.rename(columns={frame.columns[0]: "date"})
    frame["date"] = pd.to_datetime(frame["date"])
    frame = frame.merge(context, on="date", how="left").dropna(subset=["strategy_return", "benchmark_return"])
    crash_threshold = frame["benchmark_return"].quantile(0.10)
    rally_threshold = frame["benchmark_return"].quantile(0.90)
    masks = {
        "benchmark_up_days": frame["benchmark_return"] > 0,
        "benchmark_down_days": frame["benchmark_return"] < 0,
        "high_benchmark_vol": frame["spy_vol_20d_ann"] >= regime_thresholds["high_vol_spy_vol_20d_ann"],
        "benchmark_worst_decile_days": frame["benchmark_return"] <= crash_threshold,
        "benchmark_rally_days": frame["benchmark_return"] >= rally_threshold,
        "market_drawdown_regime": frame["market_regime"].eq("drawdown"),
        "market_rebound_regime": frame["market_regime"].eq("rebound"),
    }
    rows = []
    for name, mask in masks.items():
        subset = frame.loc[mask]
        rows.append(
            {
                "bucket": name,
                "observations": int(len(subset)),
                "avg_strategy_return": float(subset["strategy_return"].mean()) if len(subset) else np.nan,
                "avg_benchmark_return": float(subset["benchmark_return"].mean()) if len(subset) else np.nan,
                "hit_rate": float((subset["strategy_return"] > 0).mean()) if len(subset) else np.nan,
            }
        )
    return pd.DataFrame(rows)


candidate_results = {}
comparison_rows = []
for name, candidate in candidate_portfolios.items():
    print("Running backtest:", name)
    candidate_result = backtest_weights(backtest_spec, candidate, benchmark=spec.benchmark_ticker, repo_root=repo_root)
    candidate_result.metrics = build_metrics(candidate_result)
    candidate_results[name] = candidate_result
    row = {"candidate": name}
    for metric in ["annual_return", "sharpe", "sortino", "max_drawdown", "average_turnover", "total_return"]:
        row[metric] = candidate_result.metrics.get(metric, np.nan)
    risk = candidate_risk_logs[name]
    row["avg_active_names"] = float((candidate_result.weights > 1e-8).sum(axis=1).mean())
    row["avg_max_weight"] = float(risk["realized_max_weight"].mean()) if not risk.empty else np.nan
    row["avg_risk_multiplier"] = float(risk["risk_multiplier"].mean()) if not risk.empty else np.nan
    comparison_rows.append(row)

backtest_comparison = pd.DataFrame(comparison_rows).sort_values("sharpe", ascending=False).reset_index(drop=True)
result = candidate_results["weekly_regime_dynamic"]
metrics = result.metrics
print("Writing final weekly regime-dynamic artifacts...")
artifact_paths = write_artifacts_for_notebook(result, output_dir)

metrics_table = pd.DataFrame([{"metric": key, "value": value} for key, value in sorted(metrics.items())])
final_stress_table = stress_table(result, spec.benchmark_ticker)
final_regime_return_table = regime_return_table(result, feature_frame, spec.benchmark_ticker)

backtest_comparison.to_parquet(output_dir / "candidate_backtest_comparison.parquet", index=False)
final_stress_table.to_parquet(output_dir / "stress_table.parquet", index=False)
final_regime_return_table.to_parquet(output_dir / "regime_return_table.parquet", index=False)
risk_log.to_parquet(output_dir / "risk_scaling_log.parquet", index=False)
variant_comparison.to_parquet(output_dir / "variant_validation_comparison.parquet", index=False)

print("Candidate backtest comparison:")
display(backtest_comparison)
print("Final metrics:")
display(metrics_table)
print("Stress table:")
display(final_stress_table)
print("Regime return table:")
display(final_regime_return_table)
print("QuantStats report:", artifact_paths["quantstats_report"])

Running backtest: weekly_base_score
Running backtest: weekly_regime_dynamic
Running backtest: every_5_trading_days_regime_static
Running backtest: every_10_trading_days_regime_static
Writing final weekly regime-dynamic artifacts...


Candidate backtest comparison:


,candidate,annual_return,sharpe,sortino,max_drawdown,average_turnover,total_return,avg_active_names,avg_max_weight,avg_risk_multiplier
0,weekly_base_score,0.222210,0.763304,1.108851,-0.338950,0.204114,1.222868,24.725962,0.097532,1.000000
1,weekly_regime_dynamic,0.181672,0.665824,0.978820,-0.356773,0.138289,0.943559,24.793269,0.078377,0.815192
2,every_5_trading_days_regime_static,0.166915,0.604914,0.884193,-0.334210,0.176432,0.851853,24.325000,0.098413,1.000000
3,every_10_trading_days_regime_static,0.158682,0.580438,0.847444,-0.360467,0.217209,0.800245,24.530000,0.096537,1.000000


Final metrics:


,metric,value
0,annual_excess_return_vs_benchmark,0.066167
1,annual_return,0.181672
2,annual_volatility,0.272852
3,average_turnover,0.138289
4,benchmark_annual_return,0.115504
5,benchmark_annual_volatility,0.179651
6,benchmark_max_drawdown,-0.234240
7,benchmark_sharpe,0.642936
8,benchmark_total_return,0.545167
9,calmar,0.509208


Stress table:


,stress_metric,value
0,worst_day,-0.071858
1,worst_5_day_period,-0.130411
2,worst_month,-0.135139
3,avg_return_benchmark_worst_decile_days,-0.016517
4,avg_return_benchmark_rally_days,0.017306
5,benchmark_worst_decile_threshold,-0.003227


Regime return table:


,bucket,observations,avg_strategy_return,avg_benchmark_return,hit_rate
0,benchmark_up_days,539,0.011254,0.007839,0.855288
1,benchmark_down_days,458,-0.011480,-0.008135,0.168122
2,high_benchmark_vol,1068,0.000296,0.000163,0.214419
3,benchmark_worst_decile_days,302,-0.016517,-0.011492,0.056291
4,benchmark_rally_days,302,0.017306,0.012157,0.970199
5,market_drawdown_regime,472,-0.001601,-0.001274,0.230932
6,market_rebound_regime,99,0.002849,0.001839,0.252525


QuantStats report: /content/Portfolio-Optimization-Lib/runs/supervised_lstm_autoencoder_week7/quantstats.html


## Reusable Inference And Checkpoint

In [13]:
def make_inference_sequences(feature_frame: pd.DataFrame, X: np.ndarray, seq_len: int) -> tuple[np.ndarray, pd.DataFrame]:
    Xs: list[np.ndarray] = []
    meta: list[dict[str, object]] = []
    working = feature_frame.sort_values(["ticker", "date"]).reset_index(drop=True)
    for ticker, group in working.groupby("ticker", sort=False):
        positions = group.index.to_numpy(dtype=int)
        if len(positions) < seq_len:
            continue
        dates = pd.to_datetime(group["date"]).to_numpy()
        for end_offset in range(seq_len - 1, len(positions)):
            window_positions = positions[end_offset - seq_len + 1 : end_offset + 1]
            row = group.iloc[end_offset]
            Xs.append(X[window_positions])
            meta.append(
                {
                    "date": pd.Timestamp(dates[end_offset]),
                    "ticker": ticker,
                    "historical_vol_20d": float(np.clip(row["vol_20d"] * np.sqrt(252.0), 1e-4, None)),
                    "spy_vol_20d_ann": float(row["spy_vol_20d_ann"]),
                    "spy_drawdown_60d": float(row["spy_drawdown_60d"]),
                    "spy_momentum_20d": float(row["spy_momentum_20d"]),
                    "beta_60d_spy": float(row["beta_60d_spy"]),
                }
            )
    if not Xs:
        return np.empty((0, seq_len, len(ALL_FEATURE_NAMES)), dtype=np.float32), pd.DataFrame(meta)
    return np.stack(Xs).astype(np.float32), pd.DataFrame(meta)


def _series_from_mapping(value) -> pd.Series:
    return value if isinstance(value, pd.Series) else pd.Series(value, dtype=float)


def predict_from_prices(model_bundle: dict, prices: pd.DataFrame, dates=None, tickers=None) -> pd.DataFrame:
    model_obj = model_bundle["model"]
    model_device = model_bundle.get("device", device)
    feature_names = list(model_bundle["feature_names"])
    seq_len = int(model_bundle["seq_len"])
    horizon = int(model_bundle["horizon"])
    train_mean = _series_from_mapping(model_bundle["train_means"])
    train_std = _series_from_mapping(model_bundle["train_stds"])
    t_means = dict(model_bundle["target_means"])
    t_stds = dict(model_bundle["target_stds"])

    features = build_model_features(prices).replace([np.inf, -np.inf], np.nan)
    if tickers is not None:
        tickers_normalized = [str(ticker).upper() for ticker in tickers]
        features = features.loc[features["ticker"].isin(tickers_normalized)].copy()
    features = features.dropna(subset=feature_names).sort_values(["ticker", "date"]).reset_index(drop=True)
    if features.empty:
        return pd.DataFrame(columns=["date", "ticker", "horizon", "expected_return"])

    X = ((features[feature_names] - train_mean.loc[feature_names]) / train_std.loc[feature_names]).to_numpy(dtype=np.float32)
    X_infer, meta = make_inference_sequences(features, X, seq_len)
    if len(meta) == 0:
        return pd.DataFrame(columns=["date", "ticker", "horizon", "expected_return"])

    model_obj.eval()
    returns, alphas, vols, downside_probs, regime_probs, latent_norms = [], [], [], [], [], []
    with torch.no_grad():
        for start in range(0, len(X_infer), 4096):
            xb = torch.as_tensor(X_infer[start : start + 4096], dtype=torch.float32, device=model_device)
            output = model_obj(xb)
            returns.append(output["expected_return"].detach().cpu().numpy())
            alphas.append(output["expected_alpha"].detach().cpu().numpy())
            vols.append(output["expected_vol_log"].detach().cpu().numpy())
            downside_probs.append(torch.sigmoid(output["downside_logit"]).detach().cpu().numpy())
            regime_probs.append(torch.softmax(output["regime_logits"], dim=1).detach().cpu().numpy())
            latent_norms.append(torch.linalg.norm(output["latent"], dim=1).detach().cpu().numpy())

    scored = meta.copy()
    scored["expected_return"] = np.concatenate(returns) * t_stds["return"] + t_means["return"]
    scored["expected_alpha"] = np.concatenate(alphas) * t_stds["alpha"] + t_means["alpha"]
    vol_log = np.concatenate(vols) * t_stds["vol_log"] + t_means["vol_log"]
    scored["expected_forward_volatility"] = np.expm1(vol_log).clip(min=1e-4)
    scored["expected_volatility"] = np.clip(scored["historical_vol_20d"].to_numpy(dtype=float), 1e-4, None)
    scored["expected_downside_risk"] = np.concatenate(downside_probs)
    scored["uncertainty"] = np.abs(scored["expected_return"] - scored["expected_alpha"])
    regime_array = np.concatenate(regime_probs)
    for idx, name in enumerate(model_bundle["regime_classes"]):
        scored[f"regime_prob_{name}"] = regime_array[:, idx]
    scored["regime_confidence"] = (
        scored["regime_prob_calm_risk_on"]
        + 0.50 * scored["regime_prob_rebound"]
        - 0.50 * scored["regime_prob_high_vol"]
        - scored["regime_prob_drawdown"]
    )
    scored["base_alpha_vol_score"] = scored["expected_alpha"] / scored["expected_volatility"]
    scored["blended_regime_score"] = (
        scored["base_alpha_vol_score"]
        - DOWNSIDE_SCORE_PENALTY * scored["expected_downside_risk"]
        - VOL_SCORE_PENALTY * scored["expected_forward_volatility"]
        + REGIME_SCORE_BONUS * scored["regime_confidence"]
    )
    scored = rank_normalize_by_date(scored, "base_alpha_vol_score", "base_rank_score")
    scored = rank_normalize_by_date(scored, "blended_regime_score", "blended_rank_score")
    scored["horizon"] = horizon

    if dates is not None:
        requested_dates = pd.to_datetime(pd.Series(dates), utc=True).dt.tz_localize(None)
        scored = scored.loc[scored["date"].isin(set(requested_dates))].copy()

    return scored.loc[
        :,
        [
            "date",
            "ticker",
            "horizon",
            "expected_return",
            "expected_alpha",
            "expected_volatility",
            "expected_forward_volatility",
            "expected_downside_risk",
            "uncertainty",
            "base_alpha_vol_score",
            "blended_regime_score",
            "base_rank_score",
            "blended_rank_score",
            "regime_confidence",
            "regime_prob_calm_risk_on",
            "regime_prob_high_vol",
            "regime_prob_drawdown",
            "regime_prob_rebound",
            "spy_vol_20d_ann",
            "spy_drawdown_60d",
            "spy_momentum_20d",
            "beta_60d_spy",
        ],
    ].reset_index(drop=True)


model_config = {
    "architecture": "RegimeAwareSupervisedLSTMAutoencoder",
    "input_dim": len(ALL_FEATURE_NAMES),
    "hidden_size": HIDDEN_SIZE,
    "latent_dim": LATENT_DIM,
    "num_layers": NUM_LAYERS,
    "dropout": DROPOUT,
    "regime_classes": len(REGIME_CLASSES),
    "seq_len": SEQ_LEN,
}

checkpoint = {
    "state_dict": model.state_dict(),
    "model_state": model.state_dict(),
    "model_config": model_config,
    "selected_variant": selected_variant_name,
    "feature_names": ALL_FEATURE_NAMES,
    "base_feature_names": BASE_FEATURE_NAMES,
    "custom_feature_names": CUSTOM_FEATURE_NAMES,
    "train_means": train_means.to_dict(),
    "train_stds": train_stds.to_dict(),
    "target_means": target_means,
    "target_stds": target_stds,
    "regime_classes": REGIME_CLASSES,
    "regime_thresholds": regime_thresholds,
    "reconstruction_feature_weights": reconstruction_feature_weights.to_dict(),
    "variant_validation_comparison": variant_comparison.to_dict(orient="records"),
    "scoring_config": {
        "base_score": "expected_alpha / historical_vol_20d_ann",
        "blended_score": "base_alpha_vol_score - downside_penalty * downside_risk - vol_penalty * forward_vol + regime_bonus * regime_confidence",
        "downside_score_penalty": DOWNSIDE_SCORE_PENALTY,
        "vol_score_penalty": VOL_SCORE_PENALTY,
        "regime_score_bonus": REGIME_SCORE_BONUS,
    },
    "portfolio_config": {
        "rebalance_frequency": REBALANCE_FREQUENCY,
        "base_top_fraction": BASE_TOP_FRACTION,
        "base_max_weight": BASE_MAX_WEIGHT,
        "dynamic_min_max_weight": DYNAMIC_MIN_MAX_WEIGHT,
        "turnover_blend": TURNOVER_BLEND,
        "beta_target": BETA_TARGET,
        "dynamic_risk_scaling": True,
    },
    "horizon": HORIZON,
    "rebalance_frequency": REBALANCE_FREQUENCY,
    "random_seed": RANDOM_SEED,
}
model_artifact_path = output_dir / "supervised_lstm_autoencoder_week7.pt"
torch.save(checkpoint, model_artifact_path)
print("Saved model checkpoint:", model_artifact_path)

model_bundle = {
    "model": model,
    "device": device,
    "feature_names": ALL_FEATURE_NAMES,
    "train_means": train_means,
    "train_stds": train_stds,
    "target_means": target_means,
    "target_stds": target_stds,
    "seq_len": SEQ_LEN,
    "horizon": HORIZON,
    "regime_classes": REGIME_CLASSES,
}

submission_style_predictions = predict_from_prices(model_bundle, prices, dates=weekly_dates, tickers=validated_weights.columns)
submission_style_predictions = validate_prediction_frame(
    submission_style_predictions,
    dataset_name=DATASET_NAME,
    horizon=HORIZON,
    repo_root=repo_root,
)
print("Reusable inference predictions:", submission_style_predictions.shape)
display(submission_style_predictions.head())

Saved model checkpoint: /content/Portfolio-Optimization-Lib/runs/supervised_lstm_autoencoder_week7/supervised_lstm_autoencoder_week7.pt
Reusable inference predictions: (5198, 22)


,date,ticker,horizon,expected_return,expected_alpha,expected_volatility,expected_forward_volatility,expected_downside_risk,uncertainty,base_alpha_vol_score,...,blended_rank_score,regime_confidence,regime_prob_calm_risk_on,regime_prob_high_vol,regime_prob_drawdown,regime_prob_rebound,spy_vol_20d_ann,spy_drawdown_60d,spy_momentum_20d,beta_60d_spy
0,2022-01-07,AAPL,5,0.010556,0.009401,0.291344,0.223164,0.110177,0.001155,0.032267,...,0.20,-0.361062,0.122392,0.705070,0.144792,0.027746,0.152736,-0.024324,0.002954,1.164403
1,2022-01-07,ADBE,5,0.029642,0.021665,0.519157,0.313350,0.148287,0.007977,0.041730,...,0.04,-0.327550,0.174474,0.492132,0.281770,0.051625,0.152736,-0.024324,0.002954,2.093537
2,2022-01-07,ADI,5,0.010750,0.004796,0.242832,0.267945,0.138554,0.005954,0.019752,...,-0.04,-0.234635,0.218272,0.591196,0.168383,0.022148,0.152736,-0.024324,0.002954,1.297958
3,2022-01-07,AMAT,5,0.007514,0.002413,0.408457,0.324751,0.172684,0.005101,0.005907,...,0.68,0.370105,0.593209,0.324889,0.067740,0.014161,0.152736,-0.024324,0.002954,1.853233
4,2022-01-07,AMD,5,0.022734,0.020026,0.588302,0.378952,0.184027,0.002708,0.034041,...,0.44,0.144809,0.459627,0.314937,0.180045,0.045391,0.152736,-0.024324,0.002954,2.407294


## MLflow Logging

In [14]:
def mlflow_tracking_preflight(tracking_uri: str, timeout: float = 2.0) -> tuple[bool, str]:
    """Return quickly before MLflow starts urllib retry loops in disconnected runtimes."""
    from urllib.error import HTTPError, URLError
    from urllib.parse import urlparse
    from urllib.request import Request, urlopen
    import socket

    parsed = urlparse(tracking_uri)
    if parsed.scheme not in {"http", "https"}:
        return True, "local tracking URI"
    if not parsed.hostname:
        return False, f"remote tracking URI has no hostname: {tracking_uri}"

    proxy_env = ["HTTPS_PROXY", "HTTP_PROXY", "ALL_PROXY", "https_proxy", "http_proxy", "all_proxy"]
    configured_proxy = next((name for name in proxy_env if os.environ.get(name)), None)
    if configured_proxy:
        try:
            request = Request(tracking_uri.rstrip("/") + "/", method="GET")
            with urlopen(request, timeout=timeout):
                return True, f"{parsed.hostname} reachable through {configured_proxy}"
        except HTTPError as exc:
            return True, f"{parsed.hostname} reachable through {configured_proxy}; HTTP {exc.code}"
        except URLError as exc:
            return False, f"{parsed.hostname} not reachable through {configured_proxy} ({exc.reason})"
        except OSError as exc:
            return False, f"{parsed.hostname} not reachable through {configured_proxy} ({exc})"

    port = parsed.port or (443 if parsed.scheme == "https" else 80)
    try:
        with socket.create_connection((parsed.hostname, port), timeout=timeout):
            return True, f"{parsed.hostname}:{port} reachable"
    except OSError as exc:
        return False, f"{parsed.hostname}:{port} not reachable ({exc})"


if LOG_TO_MLFLOW:
    mlflow_layout = init_mlflow(repo_root)
    tracking_uri = mlflow_layout["tracking_uri"]
    print("MLflow tracking URI:", tracking_uri)
    tracking_ok, tracking_reason = mlflow_tracking_preflight(tracking_uri)
    if not tracking_ok:
        print("MLflow logging skipped:", tracking_reason)
        print("Set SKIP_MLFLOW=1 to suppress this cell, or connect to the MLflow/Tailscale host and rerun it.")
    else:
        try:
            with start_run(
                run_name=RUN_NAME,
                dataset_name=DATASET_NAME,
                tags={
                    "workflow": "week7_regime_supervised_lstm_autoencoder",
                    "model_family": "torch_lstm_autoencoder",
                    "prediction_horizon": str(HORIZON),
                    "rebalance_frequency": REBALANCE_FREQUENCY,
                    "selected_variant": selected_variant_name,
                },
                repo_root=repo_root,
            ):
                import mlflow

                mlflow.log_params(
                    {
                        "model_name": MODEL_NAME,
                        "dataset_name": DATASET_NAME,
                        "horizon": HORIZON,
                        "seq_len": SEQ_LEN,
                        "feature_count": len(ALL_FEATURE_NAMES),
                        "latent_dim": LATENT_DIM,
                        "hidden_size": HIDDEN_SIZE,
                        "num_layers": NUM_LAYERS,
                        "dropout": DROPOUT,
                        "learning_rate": LEARNING_RATE,
                        "weight_decay": WEIGHT_DECAY,
                        "epochs_requested": EPOCHS,
                        "variants": ",".join(VARIANT_NAMES),
                        "selected_variant": selected_variant_name,
                        "mask_rate": MASK_RATE,
                        "loss_weights": json.dumps(trained_variants[selected_variant_name]["loss_weights"], sort_keys=True),
                        "portfolio_score": "blended_regime_score",
                        "portfolio_dynamic_risk_scaling": True,
                        "portfolio_beta_target": BETA_TARGET,
                        "portfolio_top_fraction": BASE_TOP_FRACTION,
                        "portfolio_max_weight": BASE_MAX_WEIGHT,
                        "portfolio_turnover_blend": TURNOVER_BLEND,
                        "avg_active_names": float((validated_weights > 1e-8).sum(axis=1).mean()),
                        "avg_realized_max_weight": float(risk_log["realized_max_weight"].mean()),
                        "avg_risk_multiplier": float(risk_log["risk_multiplier"].mean()),
                        "cost_bps": spec.cost_bps,
                        "random_seed": RANDOM_SEED,
                        "val_return_mse": val_return_mse,
                        "val_alpha_mse": val_alpha_mse,
                        "val_downside_brier": val_downside_brier,
                        "val_regime_accuracy": val_regime_accuracy,
                    }
                )
                for _, row in final_stress_table.iterrows():
                    mlflow.log_metric(f"stress_{row['stress_metric']}", float(row["value"]))
                log_predictions(predictions)
                log_portfolio(portfolio)
                log_backtest(result)
                manifest = log_model_submission(
                    {"model_checkpoint": model_artifact_path},
                    model_name=MODEL_NAME,
                    model_family="torch",
                    feature_names=ALL_FEATURE_NAMES,
                    target=alpha_target_col,
                    horizon=HORIZON,
                    rebalance_frequency=REBALANCE_FREQUENCY,
                    preprocessing={
                        "scaler": "train_mean_std_non_benchmark_rows",
                        "target_scaler": "return_alpha_forward_vol_mean_std_non_benchmark_rows",
                        "train_means": train_means.to_dict(),
                        "train_stds": train_stds.to_dict(),
                        "target_means": target_means,
                        "target_stds": target_stds,
                        "regime_thresholds": regime_thresholds,
                        "regime_classes": REGIME_CLASSES,
                    },
                    model_config={
                        **model_config,
                        "selected_variant": selected_variant_name,
                        "mask_rate": MASK_RATE,
                        "loss_weights": trained_variants[selected_variant_name]["loss_weights"],
                        "portfolio_builder": "weights_from_regime_scores",
                        "required_functions": ["build_model_features", "predict_from_prices"],
                        "portfolio": portfolio.metadata,
                        "scoring_config": checkpoint["scoring_config"],
                    },
                    source_files=[NOTEBOOK_PATH] if NOTEBOOK_PATH.exists() else None,
                    notes="Week 7 regime-aware supervised LSTM autoencoder with downside-risk and market-regime heads, blended scoring, dynamic concentration scaling, beta targeting, and stress diagnostics.",
                )
                print("MLflow logging complete. Submission manifest keys:", sorted(manifest.keys()))
        except Exception as exc:
            print("MLflow logging skipped after error:", repr(exc))
else:
    print("MLflow logging skipped because SKIP_MLFLOW=1.")

MLflow tracking URI: https://adams-macbook-pro.tail5ddc35.ts.net
MLflow logging skipped: adams-macbook-pro.tail5ddc35.ts.net:443 not reachable ([Errno -2] Name or service not known)
Set SKIP_MLFLOW=1 to suppress this cell, or connect to the MLflow/Tailscale host and rerun it.


## Final Checks

In [15]:
assert {"total_return", "annual_return", "sharpe", "max_drawdown"}.issubset(result.metrics)
assert set(validated_weights.columns).isdisjoint({spec.benchmark_ticker}), "Benchmark ticker should not be traded."
assert validated_weights.index.is_monotonic_increasing
assert pd.Index(validated_weights.index.to_period("W-FRI")).is_unique, "Final weights should contain one row per week."
assert (validated_weights.sum(axis=1).round(6) == 1.0).all()
assert float(validated_weights.max(axis=1).max()) <= max(BASE_MAX_WEIGHT, 1.0 / len(validated_weights.columns) + 1e-6) + 1e-6
assert predictions["expected_volatility"].notna().all()
assert predictions["expected_downside_risk"].between(0.0, 1.0).all()
assert predictions["blended_rank_score"].notna().all()
assert predictions["ticker"].ne(spec.benchmark_ticker).all()
assert Path(artifact_paths["quantstats_report"]).exists()
assert model_artifact_path.exists()
assert "state_dict" in torch.load(model_artifact_path, map_location="cpu")
assert len(submission_style_predictions) == len(predictions)

print("End-to-end Week 7 regime-aware supervised LSTM autoencoder workflow validated successfully.")
print("Selected variant:", selected_variant_name)
print("Key metrics:")
for key in ["annual_return", "sharpe", "sortino", "max_drawdown", "average_turnover"]:
    print(f"  {key:<20s}: {result.metrics[key]:.6f}")
print("Average active names:", float((validated_weights > 1e-8).sum(axis=1).mean()))
print("Average max weight:", float(risk_log["realized_max_weight"].mean()))

display(result.nav.tail().to_frame("nav"))
display(result.turnover.tail().to_frame("turnover"))

End-to-end Week 7 regime-aware supervised LSTM autoencoder workflow validated successfully.
Selected variant: alpha_vol_regime
Key metrics:
  annual_return       : 0.181672
  sharpe              : 0.665824
  sortino             : 0.978820
  max_drawdown        : -0.356773
  average_turnover    : 0.138289
Average active names: 24.79326923076923
Average max weight: 0.07837666880744934


,nav
2025-12-24,195.686556
2025-12-26,195.958307
2025-12-29,195.638525
2025-12-30,195.690190
2025-12-31,194.161775


,turnover
date,
2025-11-28,0.126047
2025-12-05,0.175001
2025-12-12,0.212121
2025-12-19,0.222217
2025-12-23,0.176374
